# MiniTcl: análisis manual de scripts y análisis de expresiones con SLY
## Especificación y cuaderno de trabajo del estudiante

**Asignatura:** Compiladores · **Implementación:** Python 3.10 o posterior · **Enfoque:** orientación a objetos y `dataclasses`.

**Objetivo:** construir un lexer manual y un parser descendente recursivo para scripts MiniTcl, y un lexer con `sly.Lexer` y un parser LALR(1) con `sly.Parser` para las expresiones de `expr`. Ambos producen AST y no ejecutan el programa durante el análisis.

| Parte | Lexer | Parser |
|---|---|---|
| Scripts, comandos, palabras y cuerpos | Manual | `ScriptParser`, descendente recursivo |
| Expresiones de `expr` y condiciones de `if`, `while`, `for` | `ExprLexer(sly.Lexer)` | `ExprParser(sly.Parser)` |

Se entrega la especificación, el AST y esqueletos mínimos. **El estudiante completa las reglas a partir de la gramática.** El parser manual suministrado acepta solo palabras de un `TEXT`; el arranque de SLY acepta solo un número. No se entrega un intérprete ni un analizador completo. Las actividades se identifican con `TODO` o `NotImplementedError`; sus pruebas se activan por etapas.

El cuaderno se puede leer y ejecutar sin SLY: la demostración opcional de SLY se omite si no está instalado. Para desarrollar esa parte, instale `sly` en el entorno del kernel, por ejemplo con `%pip install sly` en una celda aparte, y reinicie el kernel si es necesario.

## 1. Alcance: un MiniTcl útil y realizable

La referencia pequeña es **Picol de antirez**, especialmente su enfoque didáctico; no se pretende comparar con todos los proyectos derivados que también se llaman Picol. Jim Tcl sirve como referencia de una implementación mucho más amplia. El tamaño de la gramática por sí solo no mide la capacidad de un Tcl: muchas ampliaciones son comandos incorporados, sin cambiar el parser de scripts.

Esta propuesta se llama **MiniTcl educativo 1.1**. Amplía el trabajo previo del curso con `for`, `foreach`, listas, diccionarios, operaciones de cadenas, arreglos de índice sencillo y una separación explícita entre AST estructural y validación de comandos.

| Obligatorio en el proyecto completo | Límite para mantener el proyecto realizable |
|---|---|
| Variables escalares y arreglos asociativos | Índices literales simples; sin sustituciones dentro del índice |
| Palabras sin agrupar, entre comillas y entre llaves | Sin expansión `{*}` ni concatenación después de cerrar una palabra agrupada |
| Interpolación y sustituciones `[...]` anidadas | Sin `eval`, `uplevel` ni generación dinámica de cuerpos de control |
| `expr`, condicionales, ciclos y procedimientos recursivos | Cuerpos y condiciones de control escritos entre llaves |
| Listas, diccionarios y operaciones sencillas de cadenas | Sin repertorio completo de Tcl ni bibliotecas de Jim Tcl |
| Ámbitos locales y declaración `global` | Sin namespaces, cierres, objetos ni trazas |
| `catch` y `error` | Sin opciones avanzadas de `return` ni de `catch` |

No se exigen GUI, sockets, procesos, archivos, extensiones nativas, bytecode ni generación para 6502 en esta entrega. El AST podrá servir después como entrada del checker, del intérprete y del generador de código.

**Tareas posibles:** factorial y Fibonacci recursivos; estadísticas de una lista; búsqueda lineal; histograma con diccionarios; tablas de multiplicar; procesamiento sencillo de texto y manejo de errores.

## 2. La decisión central: dos gramáticas y dos momentos

El parser estructural reconoce **scripts, comandos y palabras**. `if`, `set` y `proc` son nombres de comandos; no se convierten en palabras reservadas del lexer.

1. **Análisis estructural:** `Script → Command → Word → Part`. Las llaves producen una palabra literal; los corchetes contienen otro `Script`.
2. **Validación de comandos:** una pasada posterior comprueba aridad y las formas de los incorporados. En cuerpos y condiciones estáticos puede construir vistas `Script` y `Expr` adicionales.
3. **Ejecución futura:** resuelve variables, concatena partes de palabras y ejecuta comandos. Este paso no forma parte del trabajo del parser.

En `set mensaje {if algo}`, el bloque es texto; **no** es un `If`. En `while {$i < 10} {incr i}`, los argumentos son palabras literales que el comando interpreta como expresión y script, respectivamente.

Se recomienda conservar el AST estructural y asociarle un mapa de cuerpos y expresiones ya analizados. Un AST especializado con `If` o `While` puede añadirse después, pero no debe cambiar el significado de cualquier palabra entre llaves.

### Respecto de la especificación anterior del curso

Se conserva la precedencia de `expr`, incluido `-2 ** 2 = -(2 ** 2)`. Se precisan tres cambios: `#` abre comentario solo al comienzo de un comando; una palabra puede contener varias partes (`hola_$nombre`); y se conservan espacios significativos para distinguir palabras. No debe mezclarse este contrato con el lexer anterior sin adaptarlo.

### Frontera entre las dos herramientas

`ScriptParser` usa exclusivamente los tokens del lexer manual. El contenido literal de un argumento de `expr` se entrega a `parse_minitcl_expr()`, que usa `ExprLexer` y `ExprParser` de SLY. Las condiciones de `if`, `elseif`, `while` y `for` usan exactamente la misma fachada. Los cuerpos, la inicialización y el paso de `for` vuelven al parser manual.

En una expresión, el token `CMD_SUBST` guarda un `Script` construido por el parser manual al leer `[...]`. El parser SLY lo convierte en `SubstitutionExpr`; no ejecuta ese script. Por tanto, el cortocircuito sigue siendo posible.


## 3. Notación y reglas léxicas

En EBNF, `{ X }` significa repetición y `[ X ]` significa opcional. Los caracteres **entre comillas** son terminales: `"{"` representa una llave del programa, no una repetición de la notación. `ε` denota vacío.

### Contrato del lexer de scripts

El lexer debe conservar los límites de palabras. No basta con descartar todos los espacios.

| Token | Contenido / comportamiento |
|---|---|
| `TEXT` | Fragmento literal ya decodificado; incluye escapes procesados fuera de llaves |
| `VAR` | Nombre escalar; en un arreglo conserva también el índice literal |
| `BRACED` | Una palabra completa entre llaves; valor interior literal y span de toda la palabra |
| `QOPEN`, `QCLOSE` | Inicio y fin de palabra entre comillas |
| `LBRACKET`, `RBRACKET` | Abren y cierran una sustitución de comando |
| `WS` | Uno o más espacios horizontales fuera de palabras agrupadas |
| `SEP` | `;` o salto de línea fuera de llaves y comillas |
| `EOF` | Fin de la fuente, siempre presente |

`BRACED` se obtiene contando llaves anidadas; una expresión regular aislada no resuelve el balanceo general. Sus comillas y corchetes son texto. `\{` y `\}` no cambian la profundidad, y se conservan con su barra dentro de llaves. Dentro de comillas, espacios, `;` y saltos se incorporan a `TEXT`; `$` y `[` sí abren sustituciones. Una sustitución dentro de comillas vuelve temporalmente al modo script y luego restaura el modo comillas.

`{` y `"` agrupan solo al inicio de una palabra. Para simplificar, en una palabra no agrupada se rechazan llaves y comillas no escapadas; escribirlas mediante escapes. Tras cerrar una palabra agrupada solo puede seguir `WS`, `SEP`, `RBRACKET` o `EOF`.

`#` comienza un comentario solo donde podría empezar **un comando**, después de espacios iniciales. El comentario termina en el salto de línea; ese salto sigue separando comandos. `puts a#b` y `puts #dato` contienen texto, no comentarios. Al entrar en `[` empieza un nuevo contexto de comando.

Escapes obligatorios fuera de llaves: `\n`, `\t`, `\r`, `\\`, `\ `, `\$`, `\[`, `\]`, `\{`, `\}`, `\"`, `\;`, `\#`. Un escape desconocido es error en este perfil. Se normalizan CRLF/CR a LF antes del lexer; los offsets se refieren a esa fuente normalizada. **La continuación barra + salto de línea queda fuera del perfil obligatorio** y debe rechazarse explícitamente; puede agregarse como ampliación con mapa de posiciones.

### Variables y números

```ebnf
identifier = letter_or_underscore, { letter_or_underscore | digit } ;
variable   = "$", identifier, [ "(", index, ")" ]
           | "${", identifier, "}" ;
index      = index_char, { index_char } ;
```

Los identificadores usan letras ASCII. `index_char` es letra, dígito, `_`, `.`, `-` o `:`. Se aceptan `$a(3)` y `$a(nombre)`; se excluyen `$a($i)`, índices vacíos y arreglos multidimensionales. `${x}` desambigua `"${x}_fin"`. Un `$` sin nombre válido es error; para imprimirlo se escribe `\$`.

En scripts, `25` sigue siendo texto. Solo el lexer de expresiones genera `NUMBER`: `(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?`. El signo inicial es un operador; no pertenece al número. No hay hexadecimales ni números complejos en este perfil.

## 4. Gramática estructural de MiniTcl

Esta gramática opera sobre los tokens anteriores. El lexer no crea un `TEXT` vacío. `BRACED` puede tener contenido vacío y las comillas pueden delimitar una cadena vacía.

```ebnf
program       = script, EOF ;
script        = { WS | SEP }, [ command, { command_gap, command } ],
                { WS | SEP } ;
command_gap   = { WS }, SEP, { WS | SEP } ;
command       = word, { WS, word } ;
word          = BRACED | quoted_word | bare_word ;
quoted_word   = QOPEN, { part }, QCLOSE ;
bare_word     = part, { part } ;
part          = TEXT | VAR | substitution ;
substitution  = LBRACKET, script, RBRACKET ;
```

**Convención operativa para los espacios finales:** en `command`, después de consumir `WS`, solo se lee otra palabra si el siguiente token inicia `word`; si es un separador o el fin del contexto, termina el comando. El EBNF describe la secuencia válida; esta regla resuelve la decisión sobre el espacio sin retroceso.

`script` termina ante `EOF` en el nivel exterior o ante `RBRACKET` dentro de una sustitución. El método que abrió `[` consume el `RBRACKET`; el script interior lo deja pendiente. Un `RBRACKET` en el nivel exterior es error.

### ¿Por qué no usar `split()`?

```tcl
puts "Hola $nombre: [expr {2 + 3}]"
set dato {una palabra con espacios}
set ruta prefijo_${nombre}_fin
```

El primer comando tiene dos palabras, el segundo tres y el tercero tres. Los espacios dentro de agrupaciones no separan argumentos; las partes contiguas de una palabra se concatenan **al ejecutar**, sin crear nuevos argumentos.

### Recursión que sí es necesaria

`parse_substitution()` llama a `parse_script()`; este puede encontrar otra sustitución. Eso es descenso recursivo válido. No es recursividad izquierda: antes de la llamada se consumió `LBRACKET`.

## 5. Gramática de expresiones para SLY, con AST

Esta parte se implementa con **`sly.Lexer` y `sly.Parser`**, que construye un analizador LALR(1). Se utiliza una gramática por niveles, con recursividad izquierda para los operadores asociativos a la izquierda. No debe traducirse a métodos descendentes recursivos.

```ebnf
expression     = logical_or ;
logical_or     = logical_or, "||", logical_and | logical_and ;
logical_and    = logical_and, "&&", equality | equality ;
equality       = equality, ("==" | "!=" | "eq" | "ne"), relational | relational ;
relational     = relational, ("<" | "<=" | ">" | ">="), additive | additive ;
additive       = additive, ("+" | "-"), multiplicative | multiplicative ;
multiplicative = multiplicative, ("*" | "/" | "%"), unary | unary ;
unary          = ("+" | "-" | "!"), unary | power ;
power          = primary, "**", unary | primary ;
primary        = NUMBER | BOOL | VAR | STRING
               | "(", expression, ")" | CMD_SUBST ;
```

SLY recibe el fin de entrada al agotarse el iterador de tokens: **no se crea un token `EOF` para SLY**. Su símbolo inicial será `expression`; debe rechazar residuos como `2 3`. El lexer manual y su cursor sí conservan el `EOF` definido para scripts.

Los niveles de esta gramática ya definen precedencia y asociatividad. No hace falta una tabla `precedence` adicional. Si el estudiante opta por una gramática plana con tabla de precedencia, deberá justificarla y conservar exactamente los AST de las pruebas, especialmente la relación entre unarios y potencia.

| Grafía / categoría | Nombre de token recomendado para SLY |
|---|---|
| `||`, `&&` | `OR`, `AND` |
| `==`, `!=`, `eq`, `ne` | `EQ`, `NE`, `STREQ`, `STRNE` |
| `<`, `<=`, `>`, `>=` | `LT`, `LE`, `GT`, `GE` |
| `+`, `-`, `*`, `/`, `%`, `**`, `!` | `PLUS`, `MINUS`, `TIMES`, `DIVIDE`, `MOD`, `POWER`, `NOT` |
| `(`, `)` | `LPAREN`, `RPAREN` |
| Número, booleano, variable y cadena literal | `NUMBER`, `BOOL`, `VAR`, `STRING` |
| Sustitución de comando balanceada | `CMD_SUBST`, cuyo `value` será un `Script` |

El lexer de SLY debe definir los patrones largos antes de los cortos: `**` antes de `*`, `<=` antes de `<`, etc. `eq`, `ne` y los booleanos se reconocen como palabras completas. Los blancos fuera de cadenas se ignoran en el lexer de expresiones, conservando los datos necesarios para calcular posiciones. No se incorporan comentarios de script dentro de `expr`.

`BOOL` acepta `true`, `false`, `yes`, `no`, `on`, `off`; `0` y `1` son números convertibles a booleano. `STRING` es una cadena entre comillas literal, sin interpolación en este perfil; rechazar `$` o `[` no escapados dentro de ella. No hay cadenas desnudas, funciones matemáticas, ternario ni operaciones de bits en la entrega obligatoria.

### Contrato para `CMD_SUBST`

Al encontrar `[`, una acción del lexer SLY delega la lectura balanceada al analizador manual mediante un callback. Este devuelve el AST del script y el índice inmediatamente posterior al `]`. La acción devuelve un único token `CMD_SUBST` y actualiza `self.index` para continuar después del cierre. No se busca simplemente el siguiente `]` ni se usa una expresión regular para balancear todo el script. Debe respetarse el anidamiento y los modos de llaves/comillas del lexer manual.

Firma sugerida del callback:

```python
read_substitution(text, opening_index, base_offset, source_name) -> tuple[Script, int]
```

`opening_index` y el índice devuelto son locales a `text`; los spans del AST se construyen sumando `base_offset`. Al avanzar sobre una sustitución multilínea, el lexer SLY también actualiza su contador de líneas. Cada llamada anidada a la fachada de expresiones crea sus propias instancias de lexer/parser, para no sobrescribir un análisis en curso.

| Entrada | Forma del AST esperada |
|---|---|
| `2 + 3 * 4` | `Binary('+', 2, Binary('*', 3, 4))` |
| `10 - 3 - 2` | `Binary('-', Binary('-', 10, 3), 2)` |
| `2 ** 3 ** 2` | `Binary('**', 2, Binary('**', 3, 2))` |
| `-2 ** 2` | `Unary('-', Binary('**', 2, 2))` |
| `2 ** -3` | `Binary('**', 2, Unary('-', 3))` |
| `$x != 0 && 10 / $x > 1` | `Binary('&&', comparación, comparación)` |

Las comparaciones encadenadas se agrupan por la izquierda. Las acciones de SLY construyen nodos, no calculan resultados. El evaluador posterior hace el cortocircuito; no usar `eval()` de Python.

## 6. Contrato de comandos: formas que debe validar la siguiente pasada

En esta tabla, `word` representa **una sola palabra** del comando, `body` es una palabra `BRACED` reinterpretada como script y `test` es una palabra `BRACED` reinterpretada como expresión. El estudiante implementa el parser genérico primero; estas formas no son alternativas del lexer.

```ebnf
set_cmd      = "set", name, [ word ] ;
unset_cmd    = "unset", name ;
incr_cmd     = "incr", name, [ word ] ;
append_cmd   = "append", name, word, { word } ;
puts_cmd     = "puts", [ "-nonewline" ], word ;
expr_cmd     = "expr", test ;
if_cmd       = "if", test, [ "then" ], body,
               { "elseif", test, [ "then" ], body }, [ "else", body ] ;
while_cmd    = "while", test, body ;
for_cmd      = "for", body, test, body, body ;
foreach_cmd  = "foreach", identifier, word, body ;
proc_cmd     = "proc", identifier, parameters, body ;
parameters   = BRACED ;   (* su contenido: lista plana de identificadores únicos *)
return_cmd   = "return", [ word ] ;
break_cmd    = "break" ;
continue_cmd = "continue" ;
global_cmd   = "global", identifier, { identifier } ;
catch_cmd    = "catch", body, [ identifier ] ;
error_cmd    = "error", word ;
call_cmd     = word, { word } ;
```

`call_cmd` es la forma estructural general y no una alternativa que deba probarse después de fallar con un incorporado. `name` es un nombre escalar o una referencia escrita `a(indice)` con el mismo índice simple del léxico. En `set a(3) 8` no se escribe `$` en el nombre que recibe el comando.

**Perfil estático del curso:** nombres de incorporados no redefinibles; nombres de procedimientos y variables receptoras de asignación escritos literalmente; condiciones y cuerpos entre llaves; `expr` recibe exactamente una expresión entre llaves. Los nombres de comandos calculados son válidos estructuralmente pero el validador los rechaza como fuera del perfil. `set accion puts; $accion hola` ilustra esa diferencia. Los argumentos de una llamada sí pueden usar interpolación y sustituciones.

Los parámetros no tienen valores por defecto ni `args` variádico. `foreach` tiene exactamente una variable y una lista; no se exigen formas de múltiples listas. `global` se permite dentro de procedimientos y enlaza un nombre con el entorno global.

### Biblioteca mínima de datos

| Forma | Semántica que deberá tener el intérprete futuro |
|---|---|
| `list ?valor ...?` | Construye una representación de lista que preserve cada elemento |
| `llength lista` | Número de elementos |
| `lindex lista indice` | Índice entero base 0; fuera de rango devuelve cadena vacía |
| `lappend nombre ?valor ...?` | Añade elementos; crea lista vacía si no existe la variable |
| `lset nombre indice valor` | Reemplaza un elemento existente; índice fuera de rango es error |
| `dict create ?clave valor ...?` | Diccionario serializado como pares; claves repetidas conservan el último valor |
| `dict get diccionario clave` | Devuelve valor; clave ausente es error |
| `dict exists diccionario clave` | Devuelve `0` o `1` |
| `dict set nombre clave valor` | Modifica variable diccionario; la crea si no existe |
| `string length texto` | Número de caracteres Unicode |
| `string tolower texto` / `string toupper texto` | Conversión de mayúsculas/minúsculas |
| `string compare a b` | Devuelve `-1`, `0` o `1`, orden lexicográfico por puntos de código |
| `split texto ?separadores?` | Separa por cualquiera de los caracteres dados; por defecto espacio, tabulador y salto |
| `join lista ?separador?` | Une elementos; separador por defecto un espacio |

En `split`, separadores contiguos producen elementos vacíos; un separador vacío divide en caracteres. En `dict create` se exige un número par de argumentos. Los arreglos `a(k)` son variables asociativas; los diccionarios son valores. No se necesita un comando `array` para la entrega obligatoria.

Estas operaciones amplían las tareas posibles sin ampliar la gramática estructural del lenguaje.

## 7. Las listas requieren su propio lector

El contenido de una lista no es un script: `$x`, `[comando]` y `;` son datos, sin sustituciones ni separación de comandos. El salto de línea separa elementos, igual que un espacio.

```ebnf
list_value    = { LIST_WS }, [ element, { LIST_WS, element } ], { LIST_WS } ;
element       = LIST_BRACED | LIST_QUOTED | LIST_BARE ;
```

`LIST_WS` contiene uno o más blancos. El lector de listas balancea llaves, elimina agrupaciones y procesa los escapes admitidos fuera de llaves; no evalúa variables ni comandos. Tras una agrupación exige blanco o fin. En elementos no agrupados se rechazan llaves y comillas no escapadas, como simplificación coherente con el script.

Contrato esencial: `parse_list(format_list(values)) == values`. No construir listas mediante `' '.join(values)` cuando los elementos contienen espacios, llaves o cadenas vacías. El lector de parámetros de `proc` puede reutilizar el lector de listas y después exigir identificadores únicos. No se entrega su implementación.

## 8. Modelo del AST

Se entrega el modelo de datos para que la actividad se concentre en traducir la gramática. `Span` usa offsets semiabiertos `[start, end)` en la fuente normalizada e incluye el nombre de la fuente. En `BRACED`, el span de `Literal` cubre el interior y el de `Word` incluye las llaves.

| Nodo | Responsabilidad |
|---|---|
| `Script` | Secuencia de comandos |
| `Command` | Secuencia de palabras; la primera es el nombre del comando |
| `Word` | Conserva modo `bare`, `quoted` o `braced` y sus partes |
| `Literal` | Texto que no debe resolverse como variable o comando |
| `Variable` | Referencia escalar o de arreglo con índice literal |
| `CommandSubstitution` | Script cuyo resultado será parte de una palabra |
| `Constant`, `VariableExpr`, `SubstitutionExpr` | Hojas del AST de expresiones |
| `Unary`, `Binary` | Operaciones sin evaluar |

Una palabra entre llaves tiene exactamente un `Literal`, incluso si es vacío. Una palabra entre comillas vacía puede tener cero partes. Los espacios de separación y los paréntesis de agrupación no necesitan nodos propios.

In [1]:
# === MODULE: model.py | PARTE 1/2 ===
"""Modelo de datos de MiniTcl: AST, spans, tokens y diagnósticos.

Los nombres de código están en inglés y las explicaciones en español, según el
contrato de la sección 18 del cuaderno.

Los spans son intervalos semiabiertos ``[start, end)`` sobre la fuente
normalizada (CRLF/CR convertidos a LF antes de analizar).
"""

from __future__ import annotations

from dataclasses import dataclass, fields, is_dataclass
from typing import Literal as Mode


@dataclass(frozen=True)
class Span:
    """Posición semiabierta ``[start, end)`` dentro de una fuente."""

    start: int
    end: int
    source: str = "<input>"


@dataclass(frozen=True, kw_only=True)
class Node:
    span: Span


@dataclass(frozen=True)
class Literal(Node):
    text: str


@dataclass(frozen=True)
class Variable(Node):
    name: str
    index: str | None = None


@dataclass(frozen=True)
class CommandSubstitution(Node):
    script: Script


Part = Literal | Variable | CommandSubstitution


@dataclass(frozen=True)
class Word(Node):
    parts: tuple[Part, ...]
    mode: Mode["bare", "quoted", "braced"] = "bare"


@dataclass(frozen=True)
class Command(Node):
    words: tuple[Word, ...]


@dataclass(frozen=True)
class Script(Node):
    commands: tuple[Command, ...]


@dataclass(frozen=True, kw_only=True)
class Expr(Node):
    pass


@dataclass(frozen=True)
class Constant(Expr):
    value: int | float | str | bool


@dataclass(frozen=True)
class VariableExpr(Expr):
    variable: Variable


@dataclass(frozen=True)
class SubstitutionExpr(Expr):
    script: Script


@dataclass(frozen=True)
class Unary(Expr):
    operator: str
    operand: Expr


@dataclass(frozen=True)
class Binary(Expr):
    operator: str
    left: Expr
    right: Expr


def cover(first: Span, last: Span) -> Span:
    """Span que cubre dos spans del mismo documento.

    Es el helper que usa el esqueleto de ``ScriptParser`` y que el cuaderno
    nunca llegó a definir.
    """
    return Span(first.start, last.end, first.source)


def position_of(source_text: str | None, offset: int) -> tuple[int, int]:
    """Devuelve ``(linea, columna)`` con base 1 para un offset de la fuente."""
    if source_text is None:
        return 0, 0
    offset = max(0, min(offset, len(source_text)))
    line = source_text.count("\n", 0, offset) + 1
    line_start = source_text.rfind("\n", 0, offset) + 1
    return line, offset - line_start + 1


def source_line(source_text: str | None, line: int) -> str:
    """Devuelve el texto de la línea ``line`` (base 1) de la fuente."""
    if source_text is None:
        return ""
    lines = source_text.split("\n")
    if 1 <= line <= len(lines):
        return lines[line - 1]
    return ""


def format_diagnostic(
    message: str,
    span: Span,
    source_text: str | None = None,
    offset: int | None = None,
) -> str:
    """Compone el diagnóstico: fuente, línea, columna, fragmento y cursor."""
    position = span.start if offset is None else offset
    line, column = position_of(source_text, position)
    header = f"{span.source}:{line}:{column}: {message}"
    if source_text is None:
        return header
    text = source_line(source_text, line)
    if not text:
        return header
    caret = " " * max(0, column - 1) + "^"
    return f"{header}\n  {text}\n  {caret}"




### Un AST construido a mano para comprender el resultado

La siguiente celda **no analiza texto**. Muestra el AST que se espera obtener para `puts hola`. Observe que no se imprime `hola` como efecto del programa: únicamente se inspecciona su estructura.

In [2]:
def show_ast(value, indent=0):
    """Impresión estructurada sin Graphviz ni dependencias externas."""
    prefix = " " * indent
    if is_dataclass(value):
        print(prefix + type(value).__name__)
        for field in fields(value):
            if field.name == "span":
                continue
            item = getattr(value, field.name)
            print(prefix + "  " + field.name + ":")
            show_ast(item, indent + 4)
    elif isinstance(value, tuple):
        for item in value:
            show_ast(item, indent)
    else:
        print(prefix + repr(value))

example_ast = Script(
    commands=(Command(words=(
        Word(parts=(Literal(text="puts", span=Span(0, 4)),), span=Span(0, 4)),
        Word(parts=(Literal(text="hola", span=Span(5, 9)),), span=Span(5, 9)),
    ), span=Span(0, 9)),), span=Span(0, 9))
show_ast(example_ast)

Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


## 9. Infraestructura mínima: tokens, cursor y diagnósticos

El lexer deberá producir la secuencia de `Token`. No se entrega un lexer completo. Para `VAR`, `value` contiene el nombre e `index` el índice opcional; para `BRACED`, `value` es el contenido interior sin decodificar. La implementación posterior debe conservar la fuente original normalizada para reanalizar los cuerpos con offsets correctos.

El cursor permite mirar, aceptar o exigir tokens. Los métodos del estudiante deben producir `ParseError` para entradas incorrectas; `NotImplementedError` significa una actividad todavía pendiente, no un error del programa MiniTcl.

In [3]:
# === MODULE: model.py | PARTE 2/2 ===
@dataclass(frozen=True)
class Token:
    """Token del lexer manual de scripts.

    ``index`` guarda el índice literal de un arreglo (``$a(3)``) y es ``None``
    para el resto de tokens.
    """

    kind: str
    value: str
    span: Span
    index: str | None = None


class ParseError(Exception):
    """Error de análisis léxico o sintáctico con posición localizada."""

    def __init__(
        self,
        message: str,
        span: Span,
        source_text: str | None = None,
        offset: int | None = None,
    ):
        self.message = message
        self.span = span
        self.source_text = source_text
        self.offset = span.start if offset is None else offset
        self.line, self.column = position_of(source_text, self.offset)
        super().__init__(format_diagnostic(message, span, source_text, self.offset))


class TokenCursor:
    """Cursor sobre una secuencia de tokens terminada en ``EOF``."""

    def __init__(self, tokens, source: str | None = None, source_base_offset: int = 0):
        self.tokens = tuple(tokens)
        if not self.tokens or self.tokens[-1].kind != "EOF":
            raise ValueError("El lexer debe agregar EOF")
        self.position = 0
        self.source = source
        self.source_base_offset = source_base_offset

    @property
    def current(self):
        return self.tokens[self.position]

    def accept(self, kind):
        if self.current.kind != kind:
            return None
        token = self.current
        if kind != "EOF":
            self.position += 1
        return token

    def expect(self, kind):
        token = self.accept(kind)
        if token is None:
            self.fail(f"Se esperaba {kind}; se encontró {self.current.kind}", self.current.span)
        return token

    def skip(self, *kinds):
        while self.current.kind in kinds and self.current.kind != "EOF":
            self.position += 1

    def fail(self, message: str, span: Span) -> None:
        """Lanza un ``ParseError`` con línea y columna calculadas."""
        offset = span.start - self.source_base_offset
        raise ParseError(message, span, self.source, offset)


def expr_shape(node):
    """Proyección del AST de expresiones para comparar forma sin offsets."""
    if isinstance(node, Constant):
        return node.value
    if isinstance(node, Unary):
        return (node.operator, expr_shape(node.operand))
    if isinstance(node, Binary):
        return (node.operator, expr_shape(node.left), expr_shape(node.right))
    if isinstance(node, VariableExpr):
        return ("var", node.variable.name, node.variable.index)
    if isinstance(node, SubstitutionExpr):
        return ("subst", tuple(len(c.words) for c in node.script.commands))
    raise TypeError(type(node).__name__)


In [4]:
# === MODULE: script_lexer.py | PARTE 1/1 ===
"""Lexer manual de scripts MiniTcl.

El lexer es *contextual*: no puede describirse con una única expresión regular
porque el significado de ``{``, ``}``, ``"``, ``[``, ``]``, ``$`` y ``#``
depende del contexto en el que aparece. Se implementa como un autómata con una
pila de marcos:

* ``command``: texto de script. ``{`` abre una palabra entre llaves (que se
  consume entera como un token ``BRACED``), ``"`` abre una palabra entre
  comillas, ``$`` abre una variable y ``[`` abre una sustitución de comando
  (que introduce un contexto de comando nuevo).
* ``quote``: interior de una palabra entre comillas. Los espacios, ``;`` y los
  saltos de línea son texto; ``$`` y ``[`` siguen abriendo sustituciones y, al
  cerrarse, se restaura el modo comillas.

Dentro de las llaves no se decodifica ningún escape: el contenido es literal y
sólo ``\\{`` y ``\\}`` se conservan con su barra para que no alteren el balanceo.
"""


from __future__ import annotations

from dataclasses import dataclass


# --- Nombres de los tokens del contrato léxico (sección 3 del cuaderno) ------
TEXT = "TEXT"
VAR = "VAR"
BRACED = "BRACED"
QOPEN = "QOPEN"
QCLOSE = "QCLOSE"
LBRACKET = "LBRACKET"
RBRACKET = "RBRACKET"
WS = "WS"
SEP = "SEP"
EOF = "EOF"

HORIZONTAL_SPACE = " \t"

# Escapes obligatorios fuera de llaves (sección 3).
ESCAPES = {
    "n": "\n",
    "t": "\t",
    "r": "\r",
    "\\": "\\",
    " ": " ",
    "$": "$",
    "[": "[",
    "]": "]",
    "{": "{",
    "}": "}",
    '"': '"',
    ";": ";",
    "#": "#",
}

# Identificadores ASCII (sección 3): sólo letras ASCII, dígitos y "_".
IDENT_START = frozenset("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ_")
IDENT_CHARS = IDENT_START | frozenset("0123456789")
INDEX_CHARS = IDENT_CHARS | frozenset(".:-")


@dataclass
class _Frame:
    """Marco de la pila de contextos del lexer."""

    kind: str  # 'command' | 'quote'
    at_command_start: bool = True
    is_substitution: bool = False
    open_position: int = 0


class ScriptLexer:
    """Analizador léxico manual de scripts MiniTcl."""

    def __init__(self, source: str, source_name: str = "<input>", base_offset: int = 0):
        self.source = source
        self.source_name = source_name
        self.base_offset = base_offset
        self.position = 0
        self.tokens: list[Token] = []
        self.frames: list[_Frame] = [_Frame(kind="command", at_command_start=True)]
        self.stop_after_substitution = False
        self.finished = False

    # --- API pública -------------------------------------------------------
    def tokenize(self) -> list[Token]:
        """Analiza toda la fuente y devuelve los tokens (incluido ``EOF``)."""
        self.scan()
        end = self.base_offset + len(self.source)
        self.tokens.append(Token(EOF, "", Span(end, end, self.source_name)))
        return self.tokens

    def scan(self) -> list[Token]:
        """Recorre la fuente emitiendo tokens hasta el final o hasta el cierre
        de la sustitución abierta (modo ``stop_after_substitution``)."""
        while self.position < len(self.source) and not self.finished:
            if self.top().kind == "quote":
                self.scan_quote()
            else:
                self.scan_command()
        self.check_unterminated()
        return self.tokens

    # --- Utilidades de contexto --------------------------------------------
    def top(self) -> _Frame:
        return self.frames[-1]

    def command_frame(self) -> _Frame:
        """Marco de comando más cercano: donde vive la marca de inicio."""
        for frame in reversed(self.frames):
            if frame.kind == "command":
                return frame
        return self.frames[0]

    def mark_word_started(self) -> None:
        """Una palabra ha empezado: aquí ya no puede abrirse un comentario."""
        self.command_frame().at_command_start = False

    def emit(self, kind: str, value: str, start: int, end: int, index: str | None = None) -> None:
        span = Span(self.base_offset + start, self.base_offset + end, self.source_name)
        self.tokens.append(Token(kind, value, span, index))

    def span_at(self, start: int, end: int | None = None) -> Span:
        if end is None:
            end = start + 1
        return Span(self.base_offset + start, self.base_offset + end, self.source_name)

    def fail(self, message: str, start: int, end: int | None = None) -> None:
        """Lanza un ``ParseError`` con la posición exacta del problema."""
        raise ParseError(message, self.span_at(start, end), self.source, start)

    # --- Bucle principal ----------------------------------------------------
    def scan_command(self) -> None:
        char = self.source[self.position]
        if char in HORIZONTAL_SPACE:
            self.scan_whitespace()
        elif char == "\n" or char == ";":
            self.scan_separator()
        elif char == "#" and self.top().at_command_start:
            self.skip_comment()
        elif char == "{":
            self.scan_braced_word()
        elif char == '"':
            self.scan_quote_open()
        elif char == "$":
            self.scan_variable()
        elif char == "[":
            self.scan_substitution_open()
        elif char == "]":
            self.scan_substitution_close()
        else:
            self.scan_text()

    def scan_whitespace(self) -> None:
        start = self.position
        while self.position < len(self.source) and self.source[self.position] in HORIZONTAL_SPACE:
            self.position += 1
        self.emit(WS, self.source[start:self.position], start, self.position)

    def scan_separator(self) -> None:
        start = self.position
        char = self.source[self.position]
        self.position += 1
        self.emit(SEP, char, start, self.position)
        # Tras un separador puede empezar un comando nuevo (y un comentario).
        self.command_frame().at_command_start = True

    def skip_comment(self) -> None:
        """El comentario termina en el salto de línea; el salto sigue siendo SEP."""
        while self.position < len(self.source) and self.source[self.position] != "\n":
            self.position += 1

    def scan_braced_word(self) -> None:
        """Palabra entre llaves: se consume entera contando el balanceo."""
        start = self.position
        self.position += 1  # '{'
        content_start = self.position
        depth = 1
        closed = False
        while self.position < len(self.source):
            char = self.source[self.position]
            if char == "\\":
                following = self.source[self.position + 1] if self.position + 1 < len(self.source) else ""
                # Sólo \{ y \} son especiales: se conservan con su barra y no
                # alteran la profundidad. Cualquier otra secuencia se conserva
                # tal cual, porque dentro de las llaves no se decodifica nada.
                self.position += 2 if following else 1
                continue
            if char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    self.position += 1
                    closed = True
                    break
            self.position += 1
        if not closed:
            self.fail("falta la llave de cierre '}'", start)
        content = self.source[content_start:self.position - 1]
        self.mark_word_started()
        self.emit(BRACED, content, start, self.position)

    def scan_quote_open(self) -> None:
        start = self.position
        self.position += 1  # '"'
        self.mark_word_started()
        self.emit(QOPEN, '"', start, self.position)
        self.frames.append(_Frame(kind="quote", at_command_start=False, open_position=start))

    def scan_quote(self) -> None:
        """Interior de una palabra entre comillas."""
        start = self.position
        chars: list[str] = []
        while self.position < len(self.source):
            char = self.source[self.position]
            if char == '"' or char == "$" or char == "[":
                break
            if char == "\\":
                chars.append(self.decode_escape())
                continue
            chars.append(char)
            self.position += 1
        if chars:
            self.emit(TEXT, "".join(chars), start, self.position)
        if self.position >= len(self.source):
            return
        char = self.source[self.position]
        if char == '"':
            close_start = self.position
            self.position += 1
            self.emit(QCLOSE, '"', close_start, self.position)
            self.frames.pop()
        elif char == "$":
            self.scan_variable()
        else:
            self.scan_substitution_open()

    def scan_text(self) -> None:
        """Fragmento literal de una palabra sin agrupar."""
        start = self.position
        chars: list[str] = []
        while self.position < len(self.source):
            char = self.source[self.position]
            if char in HORIZONTAL_SPACE or char in "\n;":
                break
            if char in "$[":
                break
            if char == "]":
                break
            if char == "{" or char == "}":
                self.fail(
                    f"'{char}' no puede aparecer en una palabra sin agrupar; "
                    f"escríbala como '\\{char}'",
                    self.position,
                )
            if char == '"':
                self.fail(
                    "'\"' no puede aparecer en una palabra sin agrupar; "
                    "escríbala como '\\\"'",
                    self.position,
                )
            if char == "\\":
                chars.append(self.decode_escape())
                continue
            chars.append(char)
            self.position += 1
        self.mark_word_started()
        self.emit(TEXT, "".join(chars), start, self.position)

    def decode_escape(self) -> str:
        start = self.position
        self.position += 1  # '\'
        if self.position >= len(self.source):
            self.fail("escape '\\' al final de la fuente", start)
        char = self.source[self.position]
        if char == "\n":
            self.fail(
                "la continuación con barra y salto de línea está fuera del perfil obligatorio",
                start,
            )
        if char not in ESCAPES:
            self.fail(f"escape desconocido '\\{char}'", start)
        self.position += 1
        return ESCAPES[char]

    def scan_variable(self) -> None:
        """Variable escalar (``$x``), con índice (``$a(3)``) o entre llaves (``${x}``)."""
        start = self.position
        self.position += 1  # '$'
        if self.position < len(self.source) and self.source[self.position] == "{":
            self.position += 1
            name_start = self.position
            while self.position < len(self.source) and self.source[self.position] in IDENT_CHARS:
                self.position += 1
            name = self.source[name_start:self.position]
            if not name:
                self.fail("'${' sin nombre de variable", start)
            if self.position >= len(self.source) or self.source[self.position] != "}":
                self.fail("falta '}' en '${...}'", start)
            self.position += 1
            self.mark_word_started()
            self.emit(VAR, name, start, self.position)
            return
        if self.position < len(self.source) and self.source[self.position] in IDENT_START:
            name_start = self.position
            while self.position < len(self.source) and self.source[self.position] in IDENT_CHARS:
                self.position += 1
            name = self.source[name_start:self.position]
            index = None
            if self.position < len(self.source) and self.source[self.position] == "(":
                index_start = self.position + 1
                scan = index_start
                while scan < len(self.source) and self.source[scan] in INDEX_CHARS:
                    scan += 1
                if scan == index_start:
                    self.fail(
                        "índice de arreglo vacío o fuera del perfil léxico",
                        self.position,
                    )
                if scan >= len(self.source) or self.source[scan] != ")":
                    self.fail("falta ')' en la referencia a un arreglo", self.position)
                index = self.source[index_start:scan]
                self.position = scan + 1
            self.mark_word_started()
            self.emit(VAR, name, start, self.position, index=index)
            return
        self.fail("'$' debe ir seguido de un nombre de variable", start)

    def scan_substitution_open(self) -> None:
        start = self.position
        self.position += 1  # '['
        self.mark_word_started()
        self.emit(LBRACKET, "[", start, self.position)
        # Al entrar en '[' empieza un contexto de comando nuevo.
        self.frames.append(
            _Frame(kind="command", at_command_start=True, is_substitution=True, open_position=start)
        )

    def scan_substitution_close(self) -> None:
        frame = self.top()
        if not frame.is_substitution:
            self.fail("']' fuera de contexto: no hay ninguna sustitución abierta", self.position)
        start = self.position
        self.position += 1  # ']'
        self.emit(RBRACKET, "]", start, self.position)
        self.frames.pop()
        if self.stop_after_substitution and len(self.frames) == 1:
            self.finished = True

    def check_unterminated(self) -> None:
        if len(self.frames) == 1:
            return
        frame = self.top()
        if frame.kind == "quote":
            self.fail("falta la comilla de cierre '\"'", frame.open_position)
        self.fail("falta el corchete de cierre ']'", frame.open_position)


def tokenize_script(source: str, source_name: str = "<input>", base_offset: int = 0) -> list[Token]:
    """Analiza una fuente completa y devuelve sus tokens."""
    return ScriptLexer(source, source_name, base_offset).tokenize()


def show_tokens(tokens) -> None:
    """Imprime la secuencia de tokens del analizador léxico.

    Se muestran las dos familias de tokens, porque su contrato es distinto:
    los del lexer manual tienen ``kind``/``value``/``span`` y los de SLY
    ``type``/``value``/``lineno``/``index``/``end``.  El ``repr`` se recorta
    porque el token ``CMD_SUBST`` lleva dentro un ``Script`` completo.
    """
    for token in tokens:
        value = repr(token.value)
        if len(value) > 40:
            value = value[:37] + "..."
        if hasattr(token, "kind"):
            print(
                f"  {token.kind:10} {value:24} "
                f"[{token.span.start}:{token.span.end}] {token.span.source}"
            )
        else:
            print(
                f"  {token.type:10} {value:24} "
                f"línea {token.lineno} pos {token.index}"
            )


def scan_substitution(
    text: str,
    opening_index: int,
    base_offset: int = 0,
    source_name: str = "<expr>",
) -> tuple[list[Token], int]:
    """Escanea el script balanceado que empieza en ``opening_index`` (un ``[``).

    Devuelve ``(tokens_internos, indice_local)`` donde ``indice_local`` es la
    posición inmediatamente posterior al ``]`` de cierre. Los tokens internos
    terminan en ``EOF`` para que el cursor pueda consumirlos.
    """
    lexer = ScriptLexer(text, source_name, base_offset)
    lexer.position = opening_index
    lexer.stop_after_substitution = True
    lexer.scan()
    inner = list(lexer.tokens[1:-1])  # se descartan LBRACKET y RBRACKET
    end = lexer.position
    inner.append(Token(EOF, "", Span(base_offset + end, base_offset + end, source_name)))
    return inner, end


## 10. Esqueleto del parser: completar desde la gramática

Solo se resuelve el caso de una palabra que consta de **un `TEXT`**. Se deja implementada la coordinación básica de scripts y comandos para mostrar el uso del cursor. El estudiante debe ampliar `parse_word`, desarrollar los métodos pendientes y escribir el lexer.

El argumento `stop` fija el fin del contexto, pero no consume ese token. En particular, el analizador exterior exige `EOF` y el de sustitución deberá exigir `RBRACKET`. Cada iteración de un bucle tiene que consumir al menos un token o terminar: un error habitual es quedar atrapado en un bucle infinito.


In [5]:
# === MODULE: script_parser.py | PARTE 1/1 ===
"""Parser descendente recursivo de scripts MiniTcl.

Trabaja exclusivamente sobre los tokens del lexer manual (sección 4 del
cuaderno) y construye el AST estructural: ``Script → Command → Word → Part``.

El parser no sabe qué comandos existen: ``if``, ``set`` o ``proc`` son nombres
de comando, no palabras reservadas. La validación de las formas de comando
vive en ``command_analyzer.py``.
"""


from __future__ import annotations


class ScriptParser(TokenCursor):
    """Analizador sintáctico descendente recursivo de scripts."""

    WORD_START = {"TEXT", "VAR", "BRACED", "QOPEN", "LBRACKET"}
    PART_START = {"TEXT", "VAR", "LBRACKET"}
    AFTER_GROUPED = {"WS", "SEP", "RBRACKET", "EOF"}

    def parse(self) -> Script:
        result = self.parse_script(stop="EOF")
        self.expect("EOF")
        return result

    def parse_script(self, stop="EOF") -> Script:
        start = self.current.span
        commands = []
        self.skip("WS", "SEP")
        while self.current.kind != stop:
            if self.current.kind == "EOF":
                self.fail(f"Falta {stop}", self.current.span)
            commands.append(self.parse_command())
            if self.current.kind == stop:
                break
            self.expect("SEP")
            self.skip("WS", "SEP")
        end = commands[-1].span if commands else self.current.span
        return Script(commands=tuple(commands), span=cover(start, end))

    def parse_command(self) -> Command:
        words = [self.parse_word()]
        while self.accept("WS"):
            if self.current.kind not in self.WORD_START:
                break
            words.append(self.parse_word())
        return Command(words=tuple(words), span=cover(words[0].span, words[-1].span))

    def parse_word(self) -> Word:
        """Selecciona la producción ``word`` según el primer token."""
        kind = self.current.kind
        if kind == "BRACED":
            return self.parse_braced_word()
        if kind == "QOPEN":
            return self.parse_quoted_word()
        if kind in self.PART_START:
            return self.parse_bare_word()
        self.fail(
            f"Se esperaba el inicio de una palabra; se encontró {kind}",
            self.current.span,
        )

    def parse_bare_word(self) -> Word:
        """``bare_word = part, { part }``: partes contiguas de una sola palabra."""
        parts = []
        while self.current.kind in self.PART_START:
            parts.append(self.parse_part())
        if not parts:
            self.fail(
                f"Se esperaba una parte de palabra; se encontró {self.current.kind}",
                self.current.span,
            )
        return Word(
            parts=tuple(parts),
            mode="bare",
            span=cover(parts[0].span, parts[-1].span),
        )

    def parse_quoted_word(self) -> Word:
        """``quoted_word = QOPEN, { part }, QCLOSE``: admite cero partes."""
        opening = self.expect("QOPEN")
        parts = []
        while self.current.kind in self.PART_START:
            parts.append(self.parse_part())
        closing = self.expect("QCLOSE")
        self.require_word_boundary("palabra entre comillas")
        return Word(parts=tuple(parts), mode="quoted", span=cover(opening.span, closing.span))

    def parse_braced_word(self) -> Word:
        """``BRACED`` produce un único ``Literal`` con el contenido literal."""
        token = self.expect("BRACED")
        literal = Literal(
            text=token.value,
            span=Span(token.span.start + 1, token.span.end - 1, token.span.source),
        )
        self.require_word_boundary("palabra entre llaves")
        return Word(parts=(literal,), mode="braced", span=token.span)

    def parse_part(self) -> Part:
        """``part = TEXT | VAR | substitution``."""
        kind = self.current.kind
        if kind == "TEXT":
            token = self.expect("TEXT")
            return Literal(text=token.value, span=token.span)
        if kind == "VAR":
            token = self.expect("VAR")
            return Variable(name=token.value, index=token.index, span=token.span)
        if kind == "LBRACKET":
            return self.parse_substitution()
        self.fail(
            f"Se esperaba una parte de palabra; se encontró {kind}",
            self.current.span,
        )

    def parse_substitution(self) -> CommandSubstitution:
        """``substitution = LBRACKET, script, RBRACKET``."""
        opening = self.expect("LBRACKET")
        script = self.parse_script(stop="RBRACKET")
        closing = self.expect("RBRACKET")
        return CommandSubstitution(script=script, span=cover(opening.span, closing.span))

    def require_word_boundary(self, what: str) -> None:
        """Tras una palabra agrupada sólo puede venir WS, SEP, ']' o EOF."""
        if self.current.kind not in self.AFTER_GROUPED:
            self.fail(
                f"se esperaba un separador después de la {what}; "
                f"se encontró {self.current.kind}",
                self.current.span,
            )


def normalize_source(source: str) -> str:
    """Normaliza CRLF/CR a LF: los spans se refieren a esta fuente."""
    return source.replace("\r\n", "\n").replace("\r", "\n")


def parse_minitcl(
    source: str,
    source_name: str = "<input>",
    base_offset: int = 0,
    diagnostic_source: str | None = None,
) -> Script:
    """Fachada pública: lexer manual + ``ScriptParser`` descendente recursivo.

    ``base_offset`` desplaza los spans: al reanálisis de un bloque entre llaves
    se usa ``word.span.start + 1``, la posición de su primer carácter interior.
    ``diagnostic_source`` es el texto en el que están expresados esos offsets
    (normalmente el fichero completo), para que línea y columna sean correctas.
    """
    normalized = normalize_source(source)
    tokens = tokenize_script(normalized, source_name, base_offset)
    if diagnostic_source is None and base_offset == 0:
        diagnostic_source = normalized
    parser = ScriptParser(tokens, source=diagnostic_source, source_base_offset=0)
    return parser.parse()


def read_substitution(
    text: str,
    opening_index: int,
    base_offset: int = 0,
    source_name: str = "<expr>",
) -> tuple[Script, int]:
    """Callback que usa el lexer de SLY para leer una sustitución balanceada.

    Devuelve ``(Script, indice_local_inmediatamente_posterior_al_cierre)``.
    El script interior se analiza con el parser manual: así se respetan los
    modos de llaves y comillas y el anidamiento de sustituciones.
    """
    tokens, end = scan_substitution(text, opening_index, base_offset, source_name)
    parser = ScriptParser(tokens, source=text, source_base_offset=base_offset)
    return parser.parse(), end


In [6]:
# === MODULE: list_reader.py | PARTE 1/1 ===
"""Lector de listas MiniTcl (sección 7 del cuaderno).

El contenido de una lista **no es un script**: ``$x``, ``[comando]`` y ``;`` son
datos, sin sustituciones ni separación de comandos. El salto de línea separa
elementos igual que un espacio.

    list_value = { LIST_WS }, [ element, { LIST_WS, element } ], { LIST_WS } ;
    element    = LIST_BRACED | LIST_QUOTED | LIST_BARE ;

El lector balancea las llaves, elimina las agrupaciones y procesa los escapes
admitidos fuera de llaves. El contrato esencial es::

    parse_list(format_list(values)) == values
"""


from __future__ import annotations


LIST_WS = "LIST_WS"
LIST_BRACED = "LIST_BRACED"
LIST_QUOTED = "LIST_QUOTED"
LIST_BARE = "LIST_BARE"

WHITESPACE = " \t\n"

ESCAPES = {
    "n": "\n",
    "t": "\t",
    "r": "\r",
    "\\": "\\",
    " ": " ",
    "$": "$",
    "[": "[",
    "]": "]",
    "{": "{",
    "}": "}",
    '"': '"',
    ";": ";",
    "#": "#",
}


class ListReader:
    """Analizador léxico del contenido de una lista."""

    def __init__(self, text: str, source_name: str = "<list>", base_offset: int = 0):
        self.text = text
        self.source_name = source_name
        self.base_offset = base_offset
        self.position = 0

    # --- Utilidades ---------------------------------------------------------
    def fail(self, message: str, start: int, end: int | None = None) -> None:
        if end is None:
            end = start + 1
        span = Span(self.base_offset + start, self.base_offset + end, self.source_name)
        raise ParseError(message, span, self.text, start)

    # --- API ----------------------------------------------------------------
    def parse(self) -> list[str]:
        values: list[str] = []
        self.skip_whitespace()
        while self.position < len(self.text):
            values.append(self.parse_element())
            if self.position >= len(self.text):
                break
            # Tras una agrupación (o tras cualquier elemento) sólo puede venir
            # un blanco o el fin de la lista.
            if self.text[self.position] not in WHITESPACE:
                self.fail(
                    "se esperaba un blanco o el fin de la lista",
                    self.position,
                )
            self.skip_whitespace()
        return values

    def skip_whitespace(self) -> None:
        while self.position < len(self.text) and self.text[self.position] in WHITESPACE:
            self.position += 1

    def parse_element(self) -> str:
        char = self.text[self.position]
        if char == "{":
            return self.parse_braced()
        if char == '"':
            return self.parse_quoted()
        return self.parse_bare()

    def parse_braced(self) -> str:
        """Elemento entre llaves: balanceado y literal."""
        start = self.position
        self.position += 1
        content_start = self.position
        depth = 1
        closed = False
        while self.position < len(self.text):
            char = self.text[self.position]
            if char == "\\":
                following = (
                    self.text[self.position + 1] if self.position + 1 < len(self.text) else ""
                )
                self.position += 2 if following else 1
                continue
            if char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    content = self.text[content_start:self.position]
                    self.position += 1
                    closed = True
                    break
            self.position += 1
        if not closed:
            self.fail("falta la llave de cierre '}' de un elemento de lista", start)
        return content

    def parse_quoted(self) -> str:
        """Elemento entre comillas: se procesan los escapes."""
        start = self.position
        self.position += 1
        chars: list[str] = []
        while self.position < len(self.text):
            char = self.text[self.position]
            if char == '"':
                self.position += 1
                return "".join(chars)
            if char == "\\":
                chars.append(self.decode_escape())
                continue
            chars.append(char)
            self.position += 1
        self.fail("falta la comilla de cierre '\"' de un elemento de lista", start)

    def parse_bare(self) -> str:
        """Elemento desnudo: sin blancos y sin llaves ni comillas sin escapar."""
        start = self.position
        chars: list[str] = []
        while self.position < len(self.text):
            char = self.text[self.position]
            if char in WHITESPACE:
                break
            if char in "{}":
                self.fail(
                    f"'{char}' no puede aparecer en un elemento sin agrupar",
                    self.position,
                )
            if char == '"':
                self.fail(
                    "'\"' no puede aparecer en un elemento sin agrupar",
                    self.position,
                )
            if char == "\\":
                chars.append(self.decode_escape())
                continue
            chars.append(char)
            self.position += 1
        if not chars:
            self.fail("se esperaba un elemento de lista", start)
        return "".join(chars)

    def decode_escape(self) -> str:
        start = self.position
        self.position += 1
        if self.position >= len(self.text):
            self.fail("escape '\\' al final de la lista", start)
        char = self.text[self.position]
        if char not in ESCAPES:
            self.fail(f"escape desconocido '\\{char}'", start)
        self.position += 1
        return ESCAPES[char]


def parse_list(text: str, source_name: str = "<list>", base_offset: int = 0) -> list[str]:
    """Analiza el contenido de una lista y devuelve sus elementos."""
    return ListReader(text, source_name, base_offset).parse()


def format_list(values) -> str:
    """Serializa una lista de forma que ``parse_list`` la reconstruya.

    No se usa ``" ".join(values)``: los elementos con espacios, llaves o
    cadenas vacías deben conservarse.
    """
    parts = []
    for value in values:
        if value == "":
            parts.append("{}")
        elif any(char in value for char in ' \t\n{}"\\;'):
            escaped = (
                value.replace("\\", "\\\\")
                .replace('"', '\\"')
                .replace("{", "\\{")
                .replace("}", "\\}")
                .replace(";", "\\;")
            )
            parts.append('"' + escaped + '"')
        else:
            parts.append(value)
    return " ".join(parts)


In [7]:
# === MODULE: expr_lexer.py | PARTE 1/1 ===
"""Lexer de expresiones MiniTcl sobre ``sly.Lexer``.

El contrato es el de la sección 5 del cuaderno:

* Los patrones largos se definen antes de los cortos (``**`` antes de ``*``,
  ``<=`` antes de ``<``, ``!=`` antes de ``!``). El regex maestro de SLY se
  construye con las definiciones en el orden en que aparecen en el cuerpo de
  la clase, de modo que el orden es significativo.
* ``CMD_SUBST`` no se reconoce con una expresión regular: al encontrar ``[``
  se delega la lectura balanceada al lexer manual mediante un callback, que
  devuelve el ``Script`` ya analizado y el índice posterior al ``]``.
* ``VAR`` admite la forma escalar, la forma entre llaves y la forma con índice
  literal de arreglo. Como ``sly.Token`` no admite campos extra, el valor del
  token es la pareja ``(nombre, índice)``.
"""


from __future__ import annotations

from sly import Lexer


IDENT_START = frozenset("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ_")
IDENT_CHARS = IDENT_START | frozenset("0123456789")
INDEX_CHARS = IDENT_CHARS | frozenset(".:-")

ESCAPES = {
    "n": "\n",
    "t": "\t",
    "r": "\r",
    "\\": "\\",
    " ": " ",
    "$": "$",
    "[": "[",
    "]": "]",
    "{": "{",
    "}": "}",
    '"': '"',
    ";": ";",
    "#": "#",
}


class ExprLexer(Lexer):
    """Lexer de la gramática de expresiones (sección 5)."""

    tokens = {
        OR,
        AND,
        EQ,
        NE,
        STREQ,
        STRNE,
        LT,
        LE,
        GT,
        GE,
        PLUS,
        MINUS,
        TIMES,
        DIVIDE,
        MOD,
        POWER,
        NOT,
        LPAREN,
        RPAREN,
        NUMBER,
        BOOL,
        VAR,
        STRING,
        CMD_SUBST,
    }

    ignore = " \t\r\n"

    # --- Operadores: primero los patrones largos ---------------------------
    POWER = r"\*\*"
    TIMES = r"\*"
    LE = r"<="
    GE = r">="
    NE = r"!="
    NOT = r"!"
    LT = r"<"
    GT = r">"
    EQ = r"=="
    OR = r"\|\|"
    AND = r"&&"
    PLUS = r"\+"
    MINUS = r"-"
    DIVIDE = r"/"
    MOD = r"%"
    LPAREN = r"\("
    RPAREN = r"\)"

    # --- Literales ----------------------------------------------------------
    NUMBER = r"(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?"
    BOOL = r"(?:true|false|yes|no|on|off)\b"
    STREQ = r"eq\b"
    STRNE = r"ne\b"

    def __init__(
        self,
        base_offset: int = 0,
        source_name: str = "<expr>",
        read_substitution=None,
    ):
        self.base_offset = base_offset
        self.source_name = source_name
        self.read_substitution = read_substitution

    # --- Utilidades ---------------------------------------------------------
    def fail(self, message: str, index: int) -> None:
        position = self.base_offset + index
        raise ParseError(
            message,
            Span(position, position + 1, self.source_name),
            self.text if hasattr(self, "text") else None,
            index,
        )

    # --- Reglas con acción --------------------------------------------------
    @_(r"\$")
    def VAR(self, t):
        """Variable escalar ``$x``, entre llaves ``${x}`` o de arreglo ``$a(3)``."""
        text = self.text
        index = t.index + 1
        index_value = None
        if index < len(text) and text[index] == "{":
            end = index + 1
            while end < len(text) and text[end] in IDENT_CHARS:
                end += 1
            name = text[index + 1: end]
            if not name:
                self.fail("'${' sin nombre de variable", t.index)
            if end >= len(text) or text[end] != "}":
                self.fail("falta '}' en '${...}'", t.index)
            end += 1
        else:
            if index >= len(text) or text[index] not in IDENT_START:
                self.fail("'$' debe ir seguido de un nombre de variable", t.index)
            end = index
            while end < len(text) and text[end] in IDENT_CHARS:
                end += 1
            name = text[index:end]
            if end < len(text) and text[end] == "(":
                index_start = end + 1
                scan = index_start
                while scan < len(text) and text[scan] in INDEX_CHARS:
                    scan += 1
                if scan == index_start:
                    self.fail("índice de arreglo vacío o fuera del perfil léxico", end)
                if scan >= len(text) or text[scan] != ")":
                    self.fail("falta ')' en la referencia a un arreglo", end)
                index_value = text[index_start:scan]
                end = scan + 1
        self.index = end
        t.end = end
        t.value = (name, index_value)
        return t

    @_(r'"')
    def STRING(self, t):
        """Cadena literal entre comillas: sin interpolación en este perfil."""
        text = self.text
        index = t.index + 1
        chars: list[str] = []
        while index < len(text):
            char = text[index]
            if char == '"':
                index += 1
                break
            if char in "$[":
                self.fail(
                    f"'{char}' no puede aparecer sin escapar dentro de una cadena de expr",
                    index,
                )
            if char == "\\":
                following = text[index + 1] if index + 1 < len(text) else ""
                if following not in ESCAPES:
                    self.fail(f"escape desconocido '\\{following}'", index)
                chars.append(ESCAPES[following])
                index += 2
                continue
            chars.append(char)
            index += 1
        else:
            self.fail("falta la comilla de cierre '\"'", t.index)
        self.index = index
        t.end = index
        t.value = "".join(chars)
        return t

    @_(r"\[")
    def CMD_SUBST(self, t):
        """Sustitución de comando: la lee el lexer manual y devuelve un ``Script``."""
        if self.read_substitution is None:
            self.fail("no hay callback para leer la sustitución de comando", t.index)
        script, end = self.read_substitution(
            self.text, t.index, self.base_offset, self.source_name
        )
        # El lexer continúa justo después del ']' y el token cubre todo el texto.
        self.index = end
        t.end = end
        t.value = script
        self.lineno += self.text.count("\n", t.index, end)
        return t

    def error(self, token):
        position = self.base_offset + token.index
        raise ParseError(
            "carácter inválido o token aún no implementado en expr",
            Span(position, position + 1, self.source_name),
            getattr(self, "text", None),
            token.index,
        )


In [8]:
# Tokens MANUALES: esta celda no sustituye el trabajo del lexer.
# Fuente equivalente: set x 10; puts hola
starter_tokens = [
    Token("TEXT", "set", Span(0, 3)), Token("WS", " ", Span(3, 4)),
    Token("TEXT", "x", Span(4, 5)), Token("WS", " ", Span(5, 6)),
    Token("TEXT", "10", Span(6, 8)), Token("SEP", ";", Span(8, 9)),
    Token("WS", " ", Span(9, 10)), Token("TEXT", "puts", Span(10, 14)),
    Token("WS", " ", Span(14, 15)), Token("TEXT", "hola", Span(15, 19)),
    Token("EOF", "", Span(19, 19)),
]
starter_ast = ScriptParser(starter_tokens).parse()
assert len(starter_ast.commands) == 2
assert starter_ast.commands[0].words[2].parts[0].text == "10"
show_ast(starter_ast)


Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'set'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'x'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                '10'
          mode:
            'bare'
    Command
      words:
        Word
          parts:
            Literal
              text:
                'puts'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola'
          mode:
            'bare'


## 11. Contratos de los métodos pendientes

| Método | Entrada / decisión | Salida y obligación |
|---|---|---|
| `parse_word` | Mirar primer token | Elegir `BRACED`, `QOPEN` o palabra no agrupada |
| `parse_bare_word` | `TEXT`, `VAR` o `LBRACKET` | Acumular partes hasta que deje de comenzar una parte; no consumir separador |
| `parse_quoted_word` | `QOPEN` | Admitir cero partes; exigir `QCLOSE`; rechazar texto adyacente tras el cierre |
| `parse_braced_word` | `BRACED` | Un `Word(mode='braced')` con un solo `Literal`; no interpretar su contenido |
| `parse_part` | Primera clase de fragmento | Producir `Literal`, `Variable` o `CommandSubstitution` |
| `parse_substitution` | `LBRACKET` | Llamar a `parse_script(stop='RBRACKET')`, consumir cierre y construir nodo |
| `parse_command` | Inicio de palabra | Acumular palabras separadas por `WS`; no confundir partes contiguas con argumentos |

### Puntos de predicción que debe justificar

- `FIRST(word) = {BRACED, QOPEN, TEXT, VAR, LBRACKET}`.
- `FIRST(part) = {TEXT, VAR, LBRACKET}`.
- Después de `QOPEN`, `QCLOSE` permite una palabra vacía.
- Después de `WS` en un comando, mirar si empieza otra palabra; de lo contrario el espacio es final.
- Después de `LBRACKET`, un `RBRACKET` inmediato admite un script vacío.

**Actividad escrita:** explique por qué el modo del lexer forma parte del contrato. No declare que toda la sintaxis de Tcl es una gramática LL(1) sobre caracteres sin mostrar cómo se tokenizan los contextos.


## 12. Esqueleto mínimo de expresiones con SLY

Se entrega únicamente el caso de una expresión formada por un número. `ExprLexer` hereda de `sly.Lexer` y `ExprParser` de `sly.Parser`; no heredan de `TokenCursor`.

**Actividad:** incorporar el repertorio de tokens de la sección 5, definir todas las producciones con `@_()` y devolver los nodos del AST. El lexer de scripts y su parser descendente recursivo siguen siendo manuales. No reutilizar los tokens del cursor manual como si fueran tokens SLY: tienen contratos distintos.

La celda es opcional cuando SLY no está instalado. No se simula la biblioteca con clases sustitutas. El arranque solo acepta un número, por lo que las pruebas completas siguen desactivadas hasta terminar las reglas.

In [9]:
# SLY es necesario para la parte de expresiones.  Si el entorno no lo tiene,
# ejecute "%pip install sly" en una celda aparte y reinicie el kernel.
try:
    import sly
    print("sly disponible:", getattr(sly, "__version__", "instalado"))
except ImportError:  # pragma: no cover - depende del entorno
    print("SLY no está instalado: las celdas de expresiones no podrán ejecutarse.")


sly disponible: 0.5


In [10]:
# === MODULE: expr_parser.py | PARTE 1/1 ===
"""Parser LALR(1) de expresiones MiniTcl sobre ``sly.Parser``.

La gramática es la de la sección 5 del cuaderno: una gramática por niveles con
recursividad izquierda para los operadores asociativos a la izquierda. Los
niveles ya definen precedencia y asociatividad, así que no hace falta una
tabla ``precedence`` adicional. La tabla LALR(1) se genera sin conflictos
shift/reduce ni reduce/reduce.

Los identificadores de código están en inglés y los comentarios en español.
"""


from __future__ import annotations

from sly import Parser


TRUE_WORDS = {"true", "yes", "on"}
FALSE_WORDS = {"false", "no", "off"}

# Símbolos no terminales de la gramática de la sección 5. Se listan porque
# ``sly`` reutiliza el atributo ``tokens`` del parser para el iterador de
# tokens durante el análisis, y no puede usarse para distinguir un terminal.
NONTERMINALS = frozenset(
    {
        "expression",
        "logical_or",
        "logical_and",
        "equality",
        "relational",
        "additive",
        "multiplicative",
        "unary",
        "power",
        "primary",
    }
)


def number_value(raw: str) -> int | float:
    """Convierte el texto de un ``NUMBER`` a ``int`` o ``float``."""
    return float(raw) if any(char in raw for char in ".eE") else int(raw)


def bool_value(raw: str) -> bool:
    """Convierte una palabra booleana de MiniTcl a ``bool``."""
    return raw.lower() in TRUE_WORDS


class ExprParser(Parser):
    """Analizador sintáctico LALR(1) de expresiones."""

    tokens = ExprLexer.tokens
    start = "expression"

    def __init__(
        self,
        text: str = "",
        base_offset: int = 0,
        source_name: str = "<expr>",
        diagnostic_source: str | None = None,
        diagnostic_offset: int | None = None,
    ):
        self.text = text
        self.base_offset = base_offset
        self.source_name = source_name
        # La fuente de diagnóstico puede ser el fichero completo: así las
        # posiciones absolutas siguen siendo correctas al reanálisis.
        self.diagnostic_source = diagnostic_source if diagnostic_source is not None else text
        self.diagnostic_offset = base_offset if diagnostic_offset is None else diagnostic_offset

    # --- Utilidades ---------------------------------------------------------
    def span_of(self, p) -> Span:
        """Span que cubre la producción completa, desplazada por ``base_offset``."""
        return Span(
            self.base_offset + p.index,
            self.base_offset + p.end,
            self.source_name,
        )

    def operator_of(self, p) -> str:
        """Texto del token operador de una producción binaria o unaria.

        ``sly`` 0.5 no expone ``p.slice`` (eso es de PLY): la referencia al
        símbolo se toma de la lista interna de la producción. El operador es el
        único símbolo *terminal* de la producción; en las binarias ocupa la
        segunda posición y en las unarias la primera.
        """
        for symbol in p._slice:
            if symbol.type not in NONTERMINALS:
                return symbol.value
        raise AssertionError("la producción no contiene ningún token operador")

    def fail(self, message: str, start: int, end: int | None = None) -> None:
        if end is None:
            end = start + 1
        span = Span(start, end, self.source_name)
        offset = self.diagnostic_offset + (start - self.base_offset)
        raise ParseError(message, span, self.diagnostic_source, offset)

    # --- Gramática por niveles ---------------------------------------------
    @_("logical_or")
    def expression(self, p):
        return p.logical_or

    @_("logical_or OR logical_and", "logical_and")
    def logical_or(self, p):
        if len(p) == 1:
            return p.logical_and
        return Binary(
            operator=self.operator_of(p),
            left=p.logical_or,
            right=p.logical_and,
            span=self.span_of(p),
        )

    @_("logical_and AND equality", "equality")
    def logical_and(self, p):
        if len(p) == 1:
            return p.equality
        return Binary(
            operator=self.operator_of(p),
            left=p.logical_and,
            right=p.equality,
            span=self.span_of(p),
        )

    @_(
        "equality EQ relational",
        "equality NE relational",
        "equality STREQ relational",
        "equality STRNE relational",
        "relational",
    )
    def equality(self, p):
        if len(p) == 1:
            return p.relational
        return Binary(
            operator=self.operator_of(p),
            left=p.equality,
            right=p.relational,
            span=self.span_of(p),
        )

    @_(
        "relational LT additive",
        "relational LE additive",
        "relational GT additive",
        "relational GE additive",
        "additive",
    )
    def relational(self, p):
        if len(p) == 1:
            return p.additive
        return Binary(
            operator=self.operator_of(p),
            left=p.relational,
            right=p.additive,
            span=self.span_of(p),
        )

    @_("additive PLUS multiplicative", "additive MINUS multiplicative", "multiplicative")
    def additive(self, p):
        if len(p) == 1:
            return p.multiplicative
        return Binary(
            operator=self.operator_of(p),
            left=p.additive,
            right=p.multiplicative,
            span=self.span_of(p),
        )

    @_(
        "multiplicative TIMES unary",
        "multiplicative DIVIDE unary",
        "multiplicative MOD unary",
        "unary",
    )
    def multiplicative(self, p):
        if len(p) == 1:
            return p.unary
        return Binary(
            operator=self.operator_of(p),
            left=p.multiplicative,
            right=p.unary,
            span=self.span_of(p),
        )

    @_("PLUS unary", "MINUS unary", "NOT unary", "power")
    def unary(self, p):
        if len(p) == 1:
            return p.power
        return Unary(operator=self.operator_of(p), operand=p.unary, span=self.span_of(p))

    @_("primary POWER unary", "primary")
    def power(self, p):
        if len(p) == 1:
            return p.primary
        return Binary(
            operator=self.operator_of(p),
            left=p.primary,
            right=p.unary,
            span=self.span_of(p),
        )

    @_(
        "NUMBER",
        "BOOL",
        "VAR",
        "STRING",
        "LPAREN expression RPAREN",
        "CMD_SUBST",
    )
    def primary(self, p):
        if len(p) == 3:
            # Los paréntesis de agrupación no necesitan nodo propio.
            return p.expression
        symbol = p._slice[0]
        span = self.span_of(p)
        if symbol.type == "NUMBER":
            return Constant(value=number_value(symbol.value), span=span)
        if symbol.type == "BOOL":
            return Constant(value=bool_value(symbol.value), span=span)
        if symbol.type == "VAR":
            name, index = symbol.value
            return VariableExpr(
                variable=Variable(name=name, index=index, span=span),
                span=span,
            )
        if symbol.type == "STRING":
            return Constant(value=symbol.value, span=span)
        return SubstitutionExpr(script=symbol.value, span=span)

    def error(self, token):
        if token is None:
            position = self.base_offset + len(self.text)
            self.fail("fin inesperado de la expresión", position, position)
        position = self.base_offset + token.index
        self.fail(
            f"token inesperado en expr: {token.type}",
            position,
            self.base_offset + token.end,
        )


def parse_minitcl_expr(
    source: str,
    *,
    base_offset: int = 0,
    source_name: str = "<expr>",
    diagnostic_source: str | None = None,
    diagnostic_offset: int | None = None,
) -> Expr:
    """Fachada común de expresiones: ``ExprLexer`` + ``ExprParser`` de SLY.

    Recibe únicamente el contenido literal de la expresión (sin ``expr`` ni las
    llaves exteriores) y consume toda la entrada, de modo que los residuos como
    ``2 3`` se rechazan.
    """
    lexer = ExprLexer(
        base_offset=base_offset,
        source_name=source_name,
        read_substitution=read_substitution,
    )
    parser = ExprParser(
        text=source,
        base_offset=base_offset,
        source_name=source_name,
        diagnostic_source=diagnostic_source,
        diagnostic_offset=diagnostic_offset,
    )
    return parser.parse(lexer.tokenize(source))


In [11]:
# === MODULE: command_analyzer.py | PARTE 1/1 ===
"""Validación de comandos y análisis estático de cuerpos y condiciones.

El parser estructural (sección 4) reconoce scripts, comandos y palabras, pero
no sabe qué comandos existen. Este módulo hace la segunda pasada:

1. Comprueba la **forma** de cada comando (sección 6): aridad y estructura de
   los incorporados.
2. Reanaliza, sin ejecutarlos, los argumentos que lo requieren (sección 13):
   las condiciones y los cuerpos de ``expr``, ``if``, ``elseif``, ``while``,
   ``for``, ``foreach``, ``proc`` y ``catch``.

El AST estructural original se conserva: esta pasada lo *anota* con los AST
adicionales, nunca lo sustituye.
"""


from __future__ import annotations

import re
from dataclasses import dataclass


NAME_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(\([A-Za-z0-9_.:-]+\))?$")
IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

# Formas de los incorporados (sección 6 y biblioteca mínima de datos).
BUILTIN_FORMS = {
    "set": "set_cmd",
    "unset": "unset_cmd",
    "incr": "incr_cmd",
    "append": "append_cmd",
    "puts": "puts_cmd",
    "expr": "expr_cmd",
    "if": "if_cmd",
    "while": "while_cmd",
    "for": "for_cmd",
    "foreach": "foreach_cmd",
    "proc": "proc_cmd",
    "return": "return_cmd",
    "break": "break_cmd",
    "continue": "continue_cmd",
    "global": "global_cmd",
    "catch": "catch_cmd",
    "error": "error_cmd",
    "list": "list_cmd",
    "llength": "llength_cmd",
    "lindex": "lindex_cmd",
    "lappend": "lappend_cmd",
    "lset": "lset_cmd",
    "dict": "dict_cmd",
    "string": "string_cmd",
    "split": "split_cmd",
    "join": "join_cmd",
}

DICT_SUBCOMMANDS = {"create", "get", "exists", "set"}
STRING_SUBCOMMANDS = {"length", "tolower", "toupper", "compare"}


@dataclass(frozen=True)
class ArgumentAnalysis:
    """Vincula una palabra del comando con el AST adicional que produce."""

    word_index: int
    kind: str  # 'expr' | 'script' | 'list' | 'name' | 'identifier' | 'word'
    span: Span
    value: object = None


@dataclass(frozen=True)
class CommandAnalysis:
    """Resultado de validar un comando y de reanalizar sus argumentos."""

    command: Command
    form: str
    arguments: tuple[ArgumentAnalysis, ...] = ()
    substitutions: tuple["ScriptAnalysis", ...] = ()
    bodies: tuple["ScriptAnalysis", ...] = ()


@dataclass(frozen=True)
class ScriptAnalysis:
    """Resultado de validar un script completo."""

    script: Script
    commands: tuple[CommandAnalysis, ...] = ()


class _Context:
    """Datos compartidos por la validación: fuente y nombre del fichero."""

    def __init__(self, source: str | None, source_name: str):
        self.source = source
        self.source_name = source_name
        # Análisis estático de los cuerpos que se hayan reanalizado.
        self.bodies: list[ScriptAnalysis] = []

    def fail(self, message: str, span: Span) -> None:
        raise ParseError(message, span, self.source, span.start)


def literal_text(word: Word) -> str | None:
    """Texto literal de una palabra sin interpolación, o ``None`` si la tiene."""
    if word.mode == "braced":
        return word.parts[0].text
    if len(word.parts) == 1 and isinstance(word.parts[0], Literal):
        return word.parts[0].text
    return None


def analyze_command(
    command: Command,
    source: str | None = None,
    source_name: str = "<input>",
) -> CommandAnalysis:
    """Valida la forma de un comando y reanalyze sus argumentos estáticos."""
    context = _Context(source, source_name)
    words = command.words
    name = literal_text(words[0])
    if name is None:
        context.fail(
            "el nombre del comando debe ser literal: los nombres calculados "
            "están fuera del perfil estático del curso",
            words[0].span,
        )
    form = BUILTIN_FORMS.get(name, "call_cmd")
    arguments: list[ArgumentAnalysis] = []
    substitutions: list[ScriptAnalysis] = []
    _collect_substitutions(command, context, substitutions)
    handler = _HANDLERS[form]
    handler(command, name, context, arguments)
    return CommandAnalysis(
        command=command,
        form=form,
        arguments=tuple(arguments),
        substitutions=tuple(substitutions),
        bodies=tuple(context.bodies),
    )


def analyze_script(
    script: Script,
    source: str | None = None,
    source_name: str = "<input>",
) -> ScriptAnalysis:
    """Valida todos los comandos de un script, incluidas las sustituciones."""
    analyses = tuple(
        analyze_command(command, source, source_name) for command in script.commands
    )
    return ScriptAnalysis(script=script, commands=analyses)


def analyze_minitcl(source: str, source_name: str = "<input>") -> ScriptAnalysis:
    """Fachada: analiza la estructura y valida las formas de comando."""
    script = parse_minitcl(source, source_name)
    normalized = source.replace("\r\n", "\n").replace("\r", "\n")
    return analyze_script(script, normalized, source_name)


# --- Utilidades de validación ------------------------------------------------
def _expect_count(
    words: tuple[Word, ...],
    minimum: int,
    maximum: int | None,
    form: str,
    context: _Context,
) -> None:
    count = len(words)
    if count < minimum or (maximum is not None and count > maximum):
        if maximum is None:
            expected = f"al menos {minimum}"
        elif minimum == maximum:
            expected = f"{minimum}"
        else:
            expected = f"entre {minimum} y {maximum}"
        context.fail(
            f"arity mismatch in {form}: se esperaba {expected} palabra(s) "
            f"y se encontraron {count}",
            words[0].span,
        )


def _expect_name(words: tuple[Word, ...], index: int, form: str, context: _Context) -> str:
    """Nombre de variable: literal ``identificador`` o ``identificador(indice)``."""
    if index >= len(words):
        context.fail(f"arity mismatch in {form}: falta el nombre", words[0].span)
    text = literal_text(words[index])
    if text is None or not NAME_PATTERN.match(text):
        context.fail(
            f"se esperaba un nombre de variable literal en {form}",
            words[index].span,
        )
    return text


def _expect_identifier(words: tuple[Word, ...], index: int, form: str, context: _Context) -> str:
    if index >= len(words):
        context.fail(f"arity mismatch in {form}: falta el identificador", words[0].span)
    text = literal_text(words[index])
    if text is None or not IDENTIFIER_PATTERN.match(text):
        context.fail(
            f"se esperaba un identificador literal en {form}",
            words[index].span,
        )
    return text


def _expect_braced(words: tuple[Word, ...], index: int, form: str, context: _Context) -> Word:
    """Un argumento ``test``, ``body`` o ``parameters`` debe ir entre llaves."""
    if index >= len(words):
        context.fail(f"arity mismatch in {form}: falta el argumento", words[0].span)
    word = words[index]
    if word.mode != "braced":
        context.fail(
            f"se esperaba un bloque entre llaves en {form}",
            word.span,
        )
    return word


def _analyze_expression(
    word: Word,
    context: _Context,
    arguments: list[ArgumentAnalysis],
    word_index: int,
) -> Expr:
    """Entrega el contenido literal de un bloque a la fachada de SLY."""
    inner = word.parts[0].text
    expression = parse_minitcl_expr(
        inner,
        base_offset=word.span.start + 1,
        source_name=word.span.source,
        diagnostic_source=context.source,
        diagnostic_offset=word.span.start + 1,
    )
    arguments.append(
        ArgumentAnalysis(
            word_index=word_index,
            kind="expr",
            span=word.span,
            value=expression,
        )
    )
    return expression


def _analyze_body(
    word: Word,
    context: _Context,
    arguments: list[ArgumentAnalysis],
    word_index: int,
) -> Script:
    """Entrega el contenido literal de un bloque al parser manual.

    El cuerpo se analiza como script *y* se valida: un error de sintaxis o de
    forma dentro de un cuerpo estático se reporta aunque la rama no se tome.
    """
    inner = word.parts[0].text
    script = parse_minitcl(
        inner,
        word.span.source,
        base_offset=word.span.start + 1,
        diagnostic_source=context.source,
    )
    arguments.append(
        ArgumentAnalysis(word_index=word_index, kind="script", span=word.span, value=script)
    )
    context.bodies.append(analyze_script(script, context.source, word.span.source))
    return script


def _collect_substitutions(
    command: Command,
    context: _Context,
    substitutions: list[ScriptAnalysis],
) -> None:
    """Los corchetes ya producen un ``Script``: también se validan."""
    for word in command.words:
        for part in word.parts:
            if hasattr(part, "script"):
                substitutions.append(
                    analyze_script(part.script, context.source, part.script.span.source)
                )


# --- Manejadores de forma ----------------------------------------------------
def _handle_call(command, name, context, arguments):
    """``call_cmd = word, { word }``: la forma estructural general."""
    _expect_count(command.words, 1, None, "call_cmd", context)
    for index, word in enumerate(command.words):
        arguments.append(
            ArgumentAnalysis(word_index=index, kind="word", span=word.span, value=word)
        )


def _handle_set(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "set_cmd", context)
    target = _expect_name(command.words, 1, "set_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )
    if len(command.words) == 3:
        arguments.append(
            ArgumentAnalysis(
                word_index=2, kind="word", span=command.words[2].span, value=command.words[2]
            )
        )


def _handle_unset(command, name, context, arguments):
    _expect_count(command.words, 2, 2, "unset_cmd", context)
    target = _expect_name(command.words, 1, "unset_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )


def _handle_incr(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "incr_cmd", context)
    target = _expect_name(command.words, 1, "incr_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )
    if len(command.words) == 3:
        arguments.append(
            ArgumentAnalysis(
                word_index=2, kind="word", span=command.words[2].span, value=command.words[2]
            )
        )


def _handle_append(command, name, context, arguments):
    _expect_count(command.words, 3, None, "append_cmd", context)
    target = _expect_name(command.words, 1, "append_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )
    for index in range(2, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_puts(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "puts_cmd", context)
    index = 1
    if literal_text(command.words[index]) == "-nonewline":
        arguments.append(
            ArgumentAnalysis(
                word_index=index,
                kind="word",
                span=command.words[index].span,
                value="-nonewline",
            )
        )
        index += 1
    if index >= len(command.words):
        context.fail("arity mismatch in puts_cmd: falta el texto", command.words[0].span)
    arguments.append(
        ArgumentAnalysis(
            word_index=index,
            kind="word",
            span=command.words[index].span,
            value=command.words[index],
        )
    )


def _handle_expr(command, name, context, arguments):
    _expect_count(command.words, 2, 2, "expr_cmd", context)
    block = _expect_braced(command.words, 1, "expr_cmd", context)
    _analyze_expression(block, context, arguments, 1)


def _handle_if(command, name, context, arguments):
    words = command.words
    if len(words) < 3:
        context.fail(
            "arity mismatch in if_cmd: se esperaba 'if' test [then] body "
            "[elseif test [then] body]... [else body]",
            words[0].span,
        )
    index = 1
    test = _expect_braced(words, index, "if_cmd", context)
    _analyze_expression(test, context, arguments, index)
    index += 1
    if index < len(words) and literal_text(words[index]) == "then":
        index += 1
    body = _expect_braced(words, index, "if_cmd", context)
    _analyze_body(body, context, arguments, index)
    index += 1
    while index < len(words) and literal_text(words[index]) == "elseif":
        index += 1
        test = _expect_braced(words, index, "if_cmd", context)
        _analyze_expression(test, context, arguments, index)
        index += 1
        if index < len(words) and literal_text(words[index]) == "then":
            index += 1
        body = _expect_braced(words, index, "if_cmd", context)
        _analyze_body(body, context, arguments, index)
        index += 1
    if index < len(words) and literal_text(words[index]) == "else":
        index += 1
        if index >= len(words):
            context.fail("falta el cuerpo de la rama 'else'", words[0].span)
        body = _expect_braced(words, index, "if_cmd", context)
        _analyze_body(body, context, arguments, index)
        index += 1
    if index != len(words):
        context.fail(
            f"arity mismatch in if_cmd: sobran {len(words) - index} palabra(s)",
            words[index].span,
        )


def _handle_while(command, name, context, arguments):
    _expect_count(command.words, 3, 3, "while_cmd", context)
    test = _expect_braced(command.words, 1, "while_cmd", context)
    _analyze_expression(test, context, arguments, 1)
    body = _expect_braced(command.words, 2, "while_cmd", context)
    _analyze_body(body, context, arguments, 2)


def _handle_for(command, name, context, arguments):
    _expect_count(command.words, 5, 5, "for_cmd", context)
    initial = _expect_braced(command.words, 1, "for_cmd", context)
    _analyze_body(initial, context, arguments, 1)
    test = _expect_braced(command.words, 2, "for_cmd", context)
    _analyze_expression(test, context, arguments, 2)
    step = _expect_braced(command.words, 3, "for_cmd", context)
    _analyze_body(step, context, arguments, 3)
    body = _expect_braced(command.words, 4, "for_cmd", context)
    _analyze_body(body, context, arguments, 4)


def _handle_foreach(command, name, context, arguments):
    _expect_count(command.words, 4, 4, "foreach_cmd", context)
    variable = _expect_identifier(command.words, 1, "foreach_cmd", context)
    arguments.append(
        ArgumentAnalysis(
            word_index=1, kind="identifier", span=command.words[1].span, value=variable
        )
    )
    arguments.append(
        ArgumentAnalysis(
            word_index=2, kind="word", span=command.words[2].span, value=command.words[2]
        )
    )
    body = _expect_braced(command.words, 3, "foreach_cmd", context)
    _analyze_body(body, context, arguments, 3)


def _handle_proc(command, name, context, arguments):
    _expect_count(command.words, 4, 4, "proc_cmd", context)
    procedure = _expect_identifier(command.words, 1, "proc_cmd", context)
    if procedure in BUILTIN_FORMS:
        context.fail(
            f"no se puede redefinir el incorporado '{procedure}'",
            command.words[1].span,
        )
    arguments.append(
        ArgumentAnalysis(
            word_index=1, kind="identifier", span=command.words[1].span, value=procedure
        )
    )
    parameters = _expect_braced(command.words, 2, "proc_cmd", context)
    names = parse_list(parameters.parts[0].text, parameters.span.source)
    seen: set[str] = set()
    for parameter in names:
        if not IDENTIFIER_PATTERN.match(parameter):
            context.fail(
                f"el parámetro '{parameter}' no es un identificador válido",
                parameters.span,
            )
        if parameter in seen:
            context.fail(
                f"el parámetro '{parameter}' está duplicado",
                parameters.span,
            )
        seen.add(parameter)
    arguments.append(
        ArgumentAnalysis(
            word_index=2, kind="list", span=parameters.span, value=tuple(names)
        )
    )
    body = _expect_braced(command.words, 3, "proc_cmd", context)
    _analyze_body(body, context, arguments, 3)


def _handle_return(command, name, context, arguments):
    _expect_count(command.words, 1, 2, "return_cmd", context)
    for index in range(1, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_break(command, name, context, arguments):
    _expect_count(command.words, 1, 1, "break_cmd", context)


def _handle_continue(command, name, context, arguments):
    _expect_count(command.words, 1, 1, "continue_cmd", context)


def _handle_global(command, name, context, arguments):
    _expect_count(command.words, 2, None, "global_cmd", context)
    for index in range(1, len(command.words)):
        variable = _expect_identifier(command.words, index, "global_cmd", context)
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="identifier", span=command.words[index].span, value=variable
            )
        )


def _handle_catch(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "catch_cmd", context)
    body = _expect_braced(command.words, 1, "catch_cmd", context)
    _analyze_body(body, context, arguments, 1)
    if len(command.words) == 3:
        variable = _expect_identifier(command.words, 2, "catch_cmd", context)
        arguments.append(
            ArgumentAnalysis(
                word_index=2, kind="identifier", span=command.words[2].span, value=variable
            )
        )


def _handle_error(command, name, context, arguments):
    _expect_count(command.words, 2, 2, "error_cmd", context)
    arguments.append(
        ArgumentAnalysis(
            word_index=1, kind="word", span=command.words[1].span, value=command.words[1]
        )
    )


def _handle_list(command, name, context, arguments):
    _expect_count(command.words, 1, None, "list_cmd", context)
    for index in range(1, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_llength(command, name, context, arguments):
    _expect_count(command.words, 2, 2, "llength_cmd", context)
    arguments.append(
        ArgumentAnalysis(
            word_index=1, kind="word", span=command.words[1].span, value=command.words[1]
        )
    )


def _handle_lindex(command, name, context, arguments):
    _expect_count(command.words, 3, 3, "lindex_cmd", context)
    for index in (1, 2):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_lappend(command, name, context, arguments):
    _expect_count(command.words, 2, None, "lappend_cmd", context)
    target = _expect_name(command.words, 1, "lappend_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )
    for index in range(2, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_lset(command, name, context, arguments):
    _expect_count(command.words, 4, 4, "lset_cmd", context)
    target = _expect_name(command.words, 1, "lset_cmd", context)
    arguments.append(
        ArgumentAnalysis(word_index=1, kind="name", span=command.words[1].span, value=target)
    )
    for index in (2, 3):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_dict(command, name, context, arguments):
    words = command.words
    _expect_count(words, 2, None, "dict_cmd", context)
    subcommand = literal_text(words[1])
    if subcommand not in DICT_SUBCOMMANDS:
        context.fail(
            "se esperaba uno de 'create', 'get', 'exists' o 'set' tras 'dict'",
            words[1].span,
        )
    if subcommand == "create":
        _expect_count(words, 2, None, "dict_create_cmd", context)
        if (len(words) - 2) % 2 != 0:
            context.fail(
                "dict create exige un número par de argumentos (clave valor)",
                words[0].span,
            )
    elif subcommand in ("get", "exists"):
        _expect_count(words, 4, 4, f"dict_{subcommand}_cmd", context)
    else:  # dict set nombre clave valor
        _expect_count(words, 5, 5, "dict_set_cmd", context)
        _expect_name(words, 2, "dict_set_cmd", context)
    for index in range(1, len(words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=words[index].span, value=words[index]
            )
        )


def _handle_string(command, name, context, arguments):
    words = command.words
    _expect_count(words, 3, None, "string_cmd", context)
    subcommand = literal_text(words[1])
    if subcommand not in STRING_SUBCOMMANDS:
        context.fail(
            "se esperaba 'length', 'tolower', 'toupper' o 'compare' tras 'string'",
            words[1].span,
        )
    if subcommand == "compare":
        _expect_count(words, 4, 4, "string_compare_cmd", context)
    else:
        _expect_count(words, 3, 3, "string_cmd", context)
    for index in range(1, len(words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=words[index].span, value=words[index]
            )
        )


def _handle_split(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "split_cmd", context)
    for index in range(1, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


def _handle_join(command, name, context, arguments):
    _expect_count(command.words, 2, 3, "join_cmd", context)
    for index in range(1, len(command.words)):
        arguments.append(
            ArgumentAnalysis(
                word_index=index, kind="word", span=command.words[index].span,
                value=command.words[index],
            )
        )


_HANDLERS = {
    "set_cmd": _handle_set,
    "unset_cmd": _handle_unset,
    "incr_cmd": _handle_incr,
    "append_cmd": _handle_append,
    "puts_cmd": _handle_puts,
    "expr_cmd": _handle_expr,
    "if_cmd": _handle_if,
    "while_cmd": _handle_while,
    "for_cmd": _handle_for,
    "foreach_cmd": _handle_foreach,
    "proc_cmd": _handle_proc,
    "return_cmd": _handle_return,
    "break_cmd": _handle_break,
    "continue_cmd": _handle_continue,
    "global_cmd": _handle_global,
    "catch_cmd": _handle_catch,
    "error_cmd": _handle_error,
    "list_cmd": _handle_list,
    "llength_cmd": _handle_llength,
    "lindex_cmd": _handle_lindex,
    "lappend_cmd": _handle_lappend,
    "lset_cmd": _handle_lset,
    "dict_cmd": _handle_dict,
    "string_cmd": _handle_string,
    "split_cmd": _handle_split,
    "join_cmd": _handle_join,
    "call_cmd": _handle_call,
}


### Cómo traducir las producciones a reglas de SLY

Traduzca cada alternativa de la sección 5 a una producción decorada con `@_()`. En una regla binaria como `additive PLUS multiplicative`, la acción debe construir `Binary` con los dos AST y un span que cubra la operación. En una regla de paso, como `additive -> multiplicative`, devuelva el AST ya construido.

**Actividad:** explicar cómo la recursividad izquierda de `additive` produce la asociación de `10 - 3 - 2`, y cómo `power -> primary POWER unary` produce la asociación de `2 ** 3 ** 2`. Esta recursividad izquierda es válida en SLY; no debe introducirse en el parser descendente recursivo de scripts.

No se entrega una acción por cada operador: el estudiante debe completarlas. Dibuje antes los AST de `8 / 2 * 3`, `2 ** 3 ** 2`, `-(4 + 1)` y `$a(3) + [doble 2]`, sin evaluarlos.

## 13. Análisis de cuerpos y condiciones, sin ejecutarlos

El AST estructural basta para guardar el programa, pero una entrega completa del análisis sintáctico debe demostrar cómo reutilizar ambos parsers en los contextos estáticos.

| Comando | Argumentos que se vuelven a analizar |
|---|---|
| `expr` | Expresión única |
| `if` / `elseif` | Cada condición con `ExprParser` de SLY; cada cuerpo con `ScriptParser` manual |
| `while` | Condición como expresión; cuerpo como script |
| `for` | Inicio, paso y cuerpo como scripts; condición como expresión |
| `foreach` | Cuerpo como script; lista sigue siendo una palabra ordinaria |
| `proc` | Parámetros como lista de nombres; cuerpo como script |
| `catch` | Cuerpo como script |

**Actividad:** escribir `analyze_command(command)` y una estructura de resultados que vincule cada argumento con su AST adicional. Mantener el AST estructural original. En la pasada estática se pueden analizar cuerpos no ejecutados; un error de sintaxis dentro de un cuerpo estático debe reportarse aunque no se tome esa rama.

Al reanalizar un `BRACED`, la primera posición interior es `word.span.start + 1`; propague ese offset y el nombre de la fuente. No desplace las posiciones calculándolas sobre texto interpolado: el perfil exige cuerpos literales. No interprete como scripts los bloques de `set`, `puts`, `list` o los datos de `dict`.

Los corchetes ya producen un `Script` durante el análisis estructural. Visite también esos scripts al validar. Una llamada a un procedimiento definido más adelante no es por sí sola un error sintáctico.

### Fachada común para expresiones

`parse_minitcl_expr(source, *, base_offset=0, source_name='<expr>') -> Expr` integra `ExprLexer` y `ExprParser` de SLY y consume toda la entrada. No recibe `expr` ni las llaves exteriores, solo su contenido literal. El analizador de comandos la llama para `expr`, `if`, `elseif`, `while` y `for`; no debe existir un segundo parser manual para condiciones.

En `set r [expr {$x + 3}]`, el lexer manual conserva el argumento de `expr` como `BRACED`. La pasada de análisis entrega `$x + 3` a SLY y asocia su `Binary` al argumento original. En `expr {$x + [doble 2]}`, el lexer SLY devuelve `VAR`, `PLUS`, `CMD_SUBST`; ese último token contiene el `Script` producido por el analizador manual. Ningún paso consulta el valor de `x` ni ejecuta `doble`.


## 15. Programas de aceptación: tareas interesantes

Los siguientes programas son **fuentes de prueba**, no código Python ejecutable. En esta etapa deben producir AST; los resultados indicados serán comprobables cuando exista el intérprete.

### A. Recursión y sustituciones anidadas

```tcl
proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]
```

Resultado futuro: `720` (se admite representación numérica equivalente). El cuerpo de `proc` es literal hasta la pasada de análisis de comandos. La expresión del retorno contiene un script que, a su vez, contiene otra expresión.

### B. Estadísticas de una lista

```tcl
set datos [list 4 8 15 16 23 42]
set suma 0
foreach dato $datos {
    set suma [expr {$suma + $dato}]
}
set promedio [expr {$suma / [llength $datos]}]
puts "Promedio: $promedio"
```

Resultado futuro: promedio `18.0`. `datos` es una lista almacenada como valor; `foreach` no analiza su contenido como código.

### C. Histograma con diccionarios

```tcl
set conteo [dict create]
foreach palabra [split {sol luna sol mar luna sol}] {
    if {[dict exists $conteo $palabra]} {
        dict set conteo $palabra [expr {[dict get $conteo $palabra] + 1}]
    } else {
        dict set conteo $palabra 1
    }
}
puts [dict get $conteo sol]
```

Resultado futuro: `3`. Las claves son valores; el comando `dict set` recibe literalmente el nombre de la variable que modifica.

### D. Ciclo `for`, arreglo y errores

```tcl
set tabla(3) 9
for {set i 0} {$i < 5} {incr i} {
    if {$i == 2} {continue}
    puts [expr {$i * $i}]
}
set codigo [catch {expr {10 / 0}} mensaje]
puts "$codigo: $mensaje"
puts $tabla(3)
```

Resultado futuro: cuadrados `0`, `1`, `9`, `16`; código de error `1` y un diagnóstico de división por cero; finalmente `9`.

**Actividad:** elija uno, enumere todas sus invocaciones recursivas de análisis y localice las palabras cuyos contenidos deben analizarse de nuevo.

## 16. Pruebas por etapas

No basta con aceptar entradas válidas: compare la **forma** del AST, la cantidad de palabras y el contenido de los literales. Las pruebas del esqueleto se ejecutan ahora. El resto se activa cuando el estudiante implemente las dos fachadas que se indican.

In [12]:
# Pruebas del material suministrado: no requieren completar actividades.
assert ScriptParser([Token("EOF", "", Span(0, 0))]).parse().commands == ()
trailing = [Token("TEXT", "puts", Span(0, 4)),
            Token("WS", " ", Span(4, 5)), Token("EOF", "", Span(5, 5))]
assert len(ScriptParser(trailing).parse().commands[0].words) == 1
try:
    ScriptParser([Token("RBRACKET", "]", Span(0, 1)),
                  Token("EOF", "", Span(1, 1))]).parse()
except (ParseError, NotImplementedError):
    # El esqueleto aún no incorpora todos los diagnósticos.
    pass
else:
    raise AssertionError("No se debe aceptar un cierre exterior")
print("Comprobaciones básicas del esqueleto: correctas.")

Comprobaciones básicas del esqueleto: correctas.


In [13]:
# --- Las dos fachadas públicas ------------------------------------------------
#
#   parse_minitcl(source)            -> script_parser.py
#       Lexer manual + ScriptParser descendente recursivo.  Devuelve un Script.
#   parse_minitcl_expr(source, ...)  -> expr_parser.py
#       ExprLexer(sly.Lexer) + ExprParser(sly.Parser).  Devuelve un Expr.
#
# Ninguna de las dos ejecuta el programa: sólo construyen el AST.  Ambas lanzan
# ParseError ante una entrada incorrecta.

RUN_STUDENT_TESTS = True  # Activar cuando estén listas las dos fachadas.
if RUN_STUDENT_TESTS:
    script = parse_minitcl('puts "Hola $nombre"; set x {a b}')
    assert len(script.commands) == 2
    assert len(script.commands[0].words) == 2
    greeting = script.commands[0].words[1]
    assert greeting.mode == "quoted"
    assert [type(p).__name__ for p in greeting.parts] == ["Literal", "Variable"]
    block = script.commands[1].words[2]
    assert block.mode == "braced" and block.parts[0].text == "a b"
    assert expr_shape(parse_minitcl_expr("2 + 3 * 4")) == ("+", 2, ("*", 3, 4))
    assert expr_shape(parse_minitcl_expr("10 - 3 - 2")) == ("-", ("-", 10, 3), 2)
    assert expr_shape(parse_minitcl_expr("2 ** 3 ** 2")) == ("**", 2, ("**", 3, 2))
    assert expr_shape(parse_minitcl_expr("-2 ** 2")) == ("-", ("**", 2, 2))
    assert expr_shape(parse_minitcl_expr("2 ** -3")) == ("**", 2, ("-", 3))
    print("Pruebas del estudiante: correctas.")
else:
    print("Pruebas completas pendientes: habilitar RUN_STUDENT_TESTS al terminar.")


Pruebas del estudiante: correctas.


In [14]:
# === MODULE: demo.py | PARTE 1/1 ===
# --- El cuerpo del módulo empieza aquí (se convierte en la celda del cuaderno).
import sys  # el cuerpo se genera desde el cuaderno, por eso el import va aquí


def argumentos_de_cli() -> list:
    """Argumentos de la línea de órdenes (vacíos dentro del cuaderno)."""
    if "get_ipython" in globals():
        return []
    return sys.argv[1:]


def demostrar_script(fuente: str, nombre: str = "<input>") -> None:
    """Tokens y AST de un script MiniTcl."""
    print(f"=== Script {nombre} -> lexer manual -> tokens -> parser descendente -> AST ===")
    show_tokens(tokenize_script(fuente, nombre))
    print("  --- AST ---")
    show_ast(parse_minitcl(fuente, nombre))


def demostrar_expresion(expresion: str) -> None:
    """Tokens y AST de una expresión."""
    print(f"=== Expresion {expresion!r} -> sly.Lexer -> tokens -> sly.Parser (LALR) -> AST ===")
    # El lexer de SLY necesita el callback que lee la sustitución con el lexer
    # manual de scripts: ese es el puente entre las dos ramas del analizador.
    lexer = ExprLexer(read_substitution=read_substitution)
    show_tokens(list(lexer.tokenize(expresion)))
    print("  --- AST ---")
    show_ast(parse_minitcl_expr(expresion))


def demostrar_archivo(ruta: str) -> None:
    """Tokens, AST y formas de comando de un fichero MiniTcl."""
    with open(ruta, encoding="utf-8") as fichero:
        fuente = fichero.read()
    try:
        demostrar_script(fuente, ruta)
        print("  --- formas de comando ---")
        for comando in analyze_minitcl(fuente, ruta).commands:
            print(f"  {comando.form:16} {len(comando.command.words)} palabras")
    except ParseError as error:
        print()
        print("=== Diagnóstico ===")
        print(error)
        raise SystemExit(1)


_argumentos = argumentos_de_cli()
if _argumentos:
    demostrar_archivo(_argumentos[0])
else:
    demostrar_script("set x hola_${nombre}_fin")
    print()
    demostrar_expresion("$n * 2 + [doble 1]")


=== Script <input> -> lexer manual -> tokens -> parser descendente -> AST ===
  TEXT       'set'                    [0:3] <input>
  WS         ' '                      [3:4] <input>
  TEXT       'x'                      [4:5] <input>
  WS         ' '                      [5:6] <input>
  TEXT       'hola_'                  [6:11] <input>
  VAR        'nombre'                 [11:20] <input>
  TEXT       '_fin'                   [20:24] <input>
  EOF        ''                       [24:24] <input>
  --- AST ---
Script
  commands:
    Command
      words:
        Word
          parts:
            Literal
              text:
                'set'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'x'
          mode:
            'bare'
        Word
          parts:
            Literal
              text:
                'hola_'
            Variable
              name:
                'nombre'
              index:
        

In [15]:
# === MODULE: tests/test_minitcl.py | PARTE 1/1 ===
# --- Utilidades --------------------------------------------------------------
def word_modes(script):
    """Lista de listas con el modo y las partes de cada palabra."""
    return [
        [(word.mode, tuple(type(p).__name__ for p in word.parts)) for word in command.words]
        for command in script.commands
    ]


def literal_of(source, command=0, word=1):
    """Texto literal de una palabra del script."""
    return parse_minitcl(source).commands[command].words[word].parts[0].text


def kinds_of(source):
    """Secuencia de tipos de token del lexer manual."""
    return [token.kind for token in tokenize_script(source)]


# --- Material suministrado ---------------------------------------------------
def test_esqueleto_suministrado():
    """Las comprobaciones de la celda 26 del cuaderno siguen siendo válidas."""
    assert ScriptParser([Token("EOF", "", Span(0, 0))]).parse().commands == ()
    trailing = [
        Token("TEXT", "puts", Span(0, 4)),
        Token("WS", " ", Span(4, 5)),
        Token("EOF", "", Span(5, 5)),
    ]
    assert len(ScriptParser(trailing).parse().commands[0].words) == 1
    try:
        ScriptParser(
            [Token("RBRACKET", "]", Span(0, 1)), Token("EOF", "", Span(1, 1))]
        ).parse()
    except ParseError:
        pass
    else:
        raise AssertionError("No se debe aceptar un cierre exterior")


def test_pruebas_del_estudiante():
    """El bloque de la celda 27 con ``RUN_STUDENT_TESTS`` activado."""
    script = parse_minitcl('puts "Hola $nombre"; set x {a b}')
    assert len(script.commands) == 2
    assert len(script.commands[0].words) == 2
    greeting = script.commands[0].words[1]
    assert greeting.mode == "quoted"
    assert [type(p).__name__ for p in greeting.parts] == ["Literal", "Variable"]
    block = script.commands[1].words[2]
    assert block.mode == "braced" and block.parts[0].text == "a b"
    assert expr_shape(parse_minitcl_expr("2 + 3 * 4")) == ("+", 2, ("*", 3, 4))
    assert expr_shape(parse_minitcl_expr("10 - 3 - 2")) == ("-", ("-", 10, 3), 2)
    assert expr_shape(parse_minitcl_expr("2 ** 3 ** 2")) == ("**", 2, ("**", 3, 2))
    assert expr_shape(parse_minitcl_expr("-2 ** 2")) == ("-", ("**", 2, 2))
    assert expr_shape(parse_minitcl_expr("2 ** -3")) == ("**", 2, ("-", 3))


# --- Contrato léxico (sección 3) ----------------------------------------------
def test_lexer_tokens_basicos():
    assert kinds_of("set x 10") == ["TEXT", "WS", "TEXT", "WS", "TEXT", "EOF"]
    assert kinds_of("set x 10; puts hi") == [
        "TEXT", "WS", "TEXT", "WS", "TEXT", "SEP", "WS", "TEXT", "WS", "TEXT", "EOF",
    ]
    assert kinds_of("set x 10\nputs hi") == [
        "TEXT", "WS", "TEXT", "WS", "TEXT", "SEP", "TEXT", "WS", "TEXT", "EOF",
    ]
    # Un salto de línea siempre es SEP, también después de un comentario.
    assert kinds_of("# comentario\nputs hi") == ["SEP", "TEXT", "WS", "TEXT", "EOF"]


def test_show_tokens():
    """La impresión de tokens funciona con las dos familias de tokens."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        show_tokens(tokenize_script("set x 1"))
        show_tokens(list(ExprLexer().tokenize("2 + 3")))
    salida = buffer.getvalue()
    assert "TEXT" in salida and "WS" in salida and "EOF" in salida
    assert "NUMBER" in salida and "PLUS" in salida
    assert "[0:3]" in salida  # la posición del primer token del script


def test_lexer_no_crea_text_vacio():
    tokens = tokenize_script('puts ""')
    assert not [t for t in tokens if t.kind == "TEXT" and t.value == ""]


def test_lexer_hash_solo_al_inicio_de_comando():
    # 'puts a#b' y 'puts #dato' contienen texto, no comentarios.
    assert literal_of("puts a#b") == "a#b"
    assert literal_of("puts #dato") == "#dato"
    # Un comentario produce un script vacío.
    assert parse_minitcl("# comentario").commands == ()
    assert parse_minitcl("# comentario\nputs hi").commands[0].words[0].parts[0].text == "puts"


def test_lexer_escapes():
    assert literal_of(r"puts a\nb") == "a\nb"
    assert literal_of(r"puts a\tb") == "a\tb"
    assert literal_of(r"puts a\\b") == "a\\b"
    assert literal_of(r"puts a\ b") == "a b"
    assert literal_of(r"puts \$x") == "$x"
    assert literal_of(r"puts \[x") == "[x"
    assert literal_of(r"puts \{x") == "{x"
    assert literal_of(r"puts a\;b") == "a;b"
    assert literal_of(r"puts a\#b") == "a#b"
    # Dentro de las llaves no se decodifica nada.
    assert literal_of(r"puts {a\nb}") == r"a\nb"
    # Un escape desconocido es error en este perfil.
    try:
        parse_minitcl(r"puts a\qb")
    except ParseError:
        pass
    else:
        raise AssertionError("un escape desconocido debe ser error")
    # La continuación con barra y salto de línea está fuera del perfil.
    try:
        parse_minitcl("puts a\\\nb")
    except ParseError:
        pass
    else:
        raise AssertionError("la continuación de línea debe rechazarse")


def test_lexer_variables():
    script = parse_minitcl("puts $a(3) ${b} $c")
    words = script.commands[0].words
    assert len(words) == 4
    assert isinstance(words[1].parts[0], Variable)
    assert words[1].parts[0].name == "a" and words[1].parts[0].index == "3"
    assert isinstance(words[2].parts[0], Variable)
    assert words[2].parts[0].name == "b" and words[2].parts[0].index is None
    assert isinstance(words[3].parts[0], Variable) and words[3].parts[0].name == "c"
    # '${x}' desambigua "${x}_fin".
    partes = parse_minitcl('puts "${x}_fin"').commands[0].words[1].parts
    assert [type(p).__name__ for p in partes] == ["Variable", "Literal"]
    assert partes[1].text == "_fin"
    # Un '$' sin nombre válido es error.
    for source in ("puts $", "puts $1", "puts $a()", "puts $a($i)"):
        try:
            parse_minitcl(source)
        except ParseError:
            pass
        else:
            raise AssertionError(f"{source!r} debe ser error léxico")


def test_lexer_balanceo_y_mensajes():
    for source, expected in (
        ("puts {sin cierre", "falta la llave de cierre"),
        ('puts "sin cierre', "falta la comilla de cierre"),
        ("puts [set x 1", "falta el corchete de cierre"),
    ):
        try:
            parse_minitcl(source)
        except ParseError as error:
            assert expected in error.message, error.message
        else:
            raise AssertionError(f"{source!r} debe ser error")


# --- Gramática estructural (sección 4) ----------------------------------------
def test_modos_de_palabra():
    assert word_modes(parse_minitcl('puts ""')) == [
        [("bare", ("Literal",)), ("quoted", ())]
    ]
    assert word_modes(parse_minitcl("puts {}")) == [
        [("bare", ("Literal",)), ("braced", ("Literal",))]
    ]
    assert word_modes(parse_minitcl("set x hola_${nombre}_fin")) == [
        [
            ("bare", ("Literal",)),
            ("bare", ("Literal",)),
            ("bare", ("Literal", "Variable", "Literal")),
        ]
    ]


def test_espacios_significativos():
    # El primer comando tiene dos palabras, el segundo tres y el tercero tres.
    assert [len(command.words) for command in parse_minitcl(
        'puts "Hola $nombre: [expr {2 + 3}]"\nset dato {una palabra con espacios}\n'
        "set ruta prefijo_${nombre}_fin"
    ).commands] == [2, 3, 3]


def test_sustituciones_anidadas():
    script = parse_minitcl("puts [set x 1; set x]")
    substitution = script.commands[0].words[1].parts[0]
    assert isinstance(substitution, CommandSubstitution)
    assert len(substitution.script.commands) == 2
    nested = parse_minitcl("puts [list [set x] y]")
    assert isinstance(nested.commands[0].words[1].parts[0], CommandSubstitution)


def test_texto_entre_llaves_es_dato():
    script = parse_minitcl("puts {a {b} c}")
    assert script.commands[0].words[1].parts[0].text == "a {b} c"
    assert script.commands[0].words[1].mode == "braced"
    # Las barras se conservan dentro de las llaves.
    assert literal_of(r"puts {a\{b\}c}") == r"a\{b\}c"
    # Un bloque entre llaves no es un condicional.
    script = parse_minitcl("puts {if mal formado}")
    assert script.commands[0].words[1].mode == "braced"


def test_texto_adyacente_tras_palabra_agrupada():
    for source in ('puts {a}b', 'puts "a"b'):
        try:
            parse_minitcl(source)
        except ParseError:
            pass
        else:
            raise AssertionError(f"{source!r} debe ser error")


def test_cierre_fuera_de_contexto():
    try:
        parse_minitcl("puts ]")
    except ParseError:
        pass
    else:
        raise AssertionError("un cierre exterior debe ser error")


def test_comentario_no_separa_comandos():
    # El salto de línea del comentario sigue separando comandos.
    script = parse_minitcl("puts a # comentario\nputs b")
    assert len(script.commands) == 2


# --- Matriz de la sección 16 --------------------------------------------------
def test_matriz_16_positivos():
    assert parse_minitcl('puts ""').commands[0].words[1].mode == "quoted"
    assert parse_minitcl("puts {}").commands[0].words[1].mode == "braced"
    assert len(parse_minitcl("set x hola_${nombre}_fin").commands[0].words) == 3
    assert len(parse_minitcl("puts [set x 1; set x]").commands[0].words[1].parts[0].script.commands) == 2
    assert literal_of("puts {a {b} c}") == "a {b} c"
    assert literal_of(r"puts {a\{b\}c}") == r"a\{b\}c"
    assert literal_of("puts a#b") == "a#b"
    assert literal_of("puts #dato") == "#dato"
    assert parse_minitcl("# comentario").commands == ()
    assert literal_of('puts "a; b"') == "a; b"
    variable = parse_minitcl("puts $a(3)").commands[0].words[1].parts[0]
    assert isinstance(variable, Variable) and variable.index == "3"


def test_matriz_16_negativos():
    casos = [
        "puts $a($i)",
        "puts {sin cierre",
        'puts "sin cierre',
        "puts [set x 1",
        "puts {a}b",
        'puts "a"b',
        "puts ]",
    ]
    for source in casos:
        try:
            parse_minitcl(source)
        except ParseError:
            pass
        else:
            raise AssertionError(f"{source!r} debe ser un ParseError")


def test_matriz_16_en_la_etapa_apropiada():
    # Errores de aridad: el AST estructural es válido, el validador no.
    script = parse_minitcl("set x 1 2")
    assert len(script.commands[0].words) == 4
    try:
        analyze_script(script, "set x 1 2")
    except ParseError:
        pass
    else:
        raise AssertionError("set x 1 2 debe fallar en la validación")
    # 'if {1} {puts si} else' es estructuralmente válido.
    script = parse_minitcl("if {1} {puts si} else")
    try:
        analyze_script(script, "if {1} {puts si} else")
    except ParseError:
        pass
    else:
        raise AssertionError("falta el cuerpo de else debe fallar en la validación")
    # 'expr {2 + }' es un script válido con una expresión inválida.
    try:
        analyze_minitcl("expr {2 + }")
    except ParseError:
        pass
    else:
        raise AssertionError("expr {2 + } debe fallar al analizar la expresión")
    # 'puts {if mal formado}' es válido: ese argumento es texto.
    analyze_minitcl("puts {if mal formado}")


# --- Expresiones con SLY (sección 5) -----------------------------------------
def test_gramatica_sly_sin_conflictos():
    table = ExprParser._lrtable
    assert len(table.sr_conflicts) == 0, table.sr_conflicts
    assert len(table.rr_conflicts) == 0, table.rr_conflicts


def test_formas_de_expresion():
    esperados = [
        ("2 + 3 * 4", ("+", 2, ("*", 3, 4))),
        ("10 - 3 - 2", ("-", ("-", 10, 3), 2)),
        ("2 ** 3 ** 2", ("**", 2, ("**", 3, 2))),
        ("-2 ** 2", ("-", ("**", 2, 2))),
        ("2 ** -3", ("**", 2, ("-", 3))),
        ("8 / 2 * 3", ("*", ("/", 8, 2), 3)),
        ("1 < 2 == 1", ("==", ("<", 1, 2), 1)),
        ("!0", ("!", 0)),
        ("true", True),
        ("$x != 0 && 10 / $x > 1", ("&&", ("!=", ("var", "x", None), 0), (">", ("/", 10, ("var", "x", None)), 1))),
    ]
    for source, esperado in esperados:
        obtenido = expr_shape(parse_minitcl_expr(source))
        assert obtenido == esperado, f"{source}: {obtenido} != {esperado}"


def test_expresion_rechaza_residuos():
    for source in ("2 3", "2 +", "(1", "", "2 @ 3"):
        try:
            parse_minitcl_expr(source)
        except ParseError:
            pass
        else:
            raise AssertionError(f"{source!r} debe ser un ParseError")


def test_sustitucion_en_expresion():
    expression = parse_minitcl_expr("[doble 2]")
    assert isinstance(expression, SubstitutionExpr)
    assert len(expression.script.commands) == 1
    # Una sustitución dentro de una expresión conserva el cortocircuito: el
    # AST no consulta ningún valor.
    expression = parse_minitcl_expr("0 && [error inesperado]")
    assert expr_shape(expression) == ("&&", 0, ("subst", (2,)))


# --- Validación de comandos (sección 6) --------------------------------------
def test_formas_de_comando():
    analysis = analyze_minitcl("set x 1")
    assert analysis.commands[0].form == "set_cmd"
    assert analysis.commands[0].arguments[0].kind == "name"
    assert analysis.commands[0].arguments[0].value == "x"
    assert analyze_minitcl("puts hi").commands[0].form == "puts_cmd"
    assert analyze_minitcl("puts -nonewline hi").commands[0].form == "puts_cmd"
    assert analyze_minitcl("miProc 1 2").commands[0].form == "call_cmd"


def test_nombre_calculado_fuera_de_perfil():
    # 'set accion puts; $accion hola' es estructuralmente válido...
    script = parse_minitcl("set accion puts; $accion hola")
    assert len(script.commands) == 2
    # ... pero el validador lo rechaza.
    try:
        analyze_script(script, "set accion puts; $accion hola")
    except ParseError:
        pass
    else:
        raise AssertionError("un nombre de comando calculado está fuera del perfil")


def test_parametros_duplicados():
    try:
        analyze_minitcl("proc f {x x} {return $x}")
    except ParseError as error:
        assert "duplicado" in error.message
    else:
        raise AssertionError("los parámetros duplicados deben ser error")


def test_incorporados_no_redefinibles():
    try:
        analyze_minitcl("proc set {x} {return $x}")
    except ParseError as error:
        assert "redefinir" in error.message
    else:
        raise AssertionError("no se puede redefinir un incorporado")


def test_cuerpo_estatico_incorrecto_se_reporta():
    # Aunque la rama no se tome, el error de sintaxis del cuerpo se reporta.
    try:
        analyze_minitcl("if {1} {puts sin cierre")
    except ParseError:
        pass
    else:
        raise AssertionError("el cuerpo estático debe analizarse")


# --- Lector de listas (sección 7) ---------------------------------------------
def test_contrato_de_listas():
    valores = [["a", "b"], [], [""], ["a b"], ["{x}"], ['say "hi"'], ["a;b"], ["a\\b"]]
    for value in valores:
        assert parse_list(format_list(value)) == value, value


def test_lista_no_es_script():
    # '$x', '[cmd]' y ';' son datos dentro de una lista.
    assert parse_list("$x [cmd] ; y") == ["$x", "[cmd]", ";", "y"]


def test_lector_de_listas_rechaza_agrupaciones_pegadas():
    for source in ("{a}b", '"a"b'):
        try:
            parse_list(source)
        except ParseError:
            pass
        else:
            raise AssertionError(f"{source!r} debe ser error")


# --- Programas de aceptación (sección 15) ------------------------------------
PROGRAMAS = {
    "A": """proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]""",
    "B": """set datos [list 4 8 15 16 23 42]
set suma 0
foreach dato $datos {
    set suma [expr {$suma + $dato}]
}
set promedio [expr {$suma / [llength $datos]}]
puts "Promedio: $promedio\"""",
    "C": """set conteo [dict create]
foreach palabra [split {sol luna sol mar luna sol}] {
    if {[dict exists $conteo $palabra]} {
        dict set conteo $palabra [expr {[dict get $conteo $palabra] + 1}]
    } else {
        dict set conteo $palabra 1
    }
}
puts [dict get $conteo sol]""",
    "D": """set tabla(3) 9
for {set i 0} {$i < 5} {incr i} {
    if {$i == 2} {continue}
    puts [expr {$i * $i}]
}
set codigo [catch {expr {10 / 0}} mensaje]
puts "$codigo: $mensaje"
puts $tabla(3)""",
}


def test_programas_de_aceptacion():
    for clave, fuente in PROGRAMAS.items():
        analysis = analyze_minitcl(fuente, f"programa_{clave}")
        assert analysis.commands, clave


def test_analisis_estatico_del_programa_a():
    """En el programa A se reanalizan el cuerpo, la condición y las expresiones."""
    analysis = analyze_minitcl(PROGRAMAS["A"], "programa_A")
    procedure = analysis.commands[0]
    assert procedure.form == "proc_cmd"
    cuerpo = [a for a in procedure.arguments if a.kind == "script"]
    assert cuerpo, "el cuerpo del procedimiento debe analizarse como script"
    assert procedure.bodies, "el cuerpo se valida como script"
    # Dentro del cuerpo, el 'if' tiene su condición analizada como expresión.
    inner_if = procedure.bodies[0].commands[0]
    assert inner_if.form == "if_cmd"
    assert [a for a in inner_if.arguments if a.kind == "expr"], "la condición se analiza"
    assert inner_if.bodies, "el cuerpo del 'if' se valida"


# --- Diagnósticos (sección 17) -----------------------------------------------
def test_diagnostico_con_linea_y_columna():
    try:
        parse_minitcl("puts hola\nputs {sin cierre")
    except ParseError as error:
        assert error.line == 2, error.line
        assert error.column == 6, error.column
        assert "sin cierre" in str(error)
        assert "^" in str(error)
    else:
        raise AssertionError("se esperaba un ParseError")


def test_cover():
    span = cover(Span(0, 3, "f"), Span(7, 9, "f"))
    assert span == Span(0, 9, "f")


# --- Ejecutor ----------------------------------------------------------------
def main() -> int:
    pruebas = [
        (nombre, funcion)
        for nombre, funcion in sorted(globals().items())
        if nombre.startswith("test_") and callable(funcion)
    ]
    fallos = 0
    for nombre, funcion in pruebas:
        try:
            funcion()
        except Exception as error:  # noqa: BLE001 - se informa de cada fallo
            fallos += 1
            print(f"FALLO  {nombre}: {type(error).__name__}: {error}")
        else:
            print(f"OK     {nombre}")
    print(f"\n{len(pruebas) - fallos}/{len(pruebas)} pruebas correctas.")
    return 1 if fallos else 0


# En el cuaderno la celda ejecuta las pruebas al final; como módulo, el
# fichero se ejecuta con "python3 tests/test_minitcl.py" (véase el sufijo de
# exportación en tools/layout.py).
if __name__ == "__main__" and "get_ipython" in globals():
    main()


OK     test_analisis_estatico_del_programa_a
OK     test_cierre_fuera_de_contexto
OK     test_comentario_no_separa_comandos
OK     test_contrato_de_listas
OK     test_cover
OK     test_cuerpo_estatico_incorrecto_se_reporta
OK     test_diagnostico_con_linea_y_columna
OK     test_espacios_significativos
OK     test_esqueleto_suministrado
OK     test_expresion_rechaza_residuos
OK     test_formas_de_comando
OK     test_formas_de_expresion
OK     test_gramatica_sly_sin_conflictos
OK     test_incorporados_no_redefinibles
OK     test_lector_de_listas_rechaza_agrupaciones_pegadas
OK     test_lexer_balanceo_y_mensajes
OK     test_lexer_escapes
OK     test_lexer_hash_solo_al_inicio_de_comando
OK     test_lexer_no_crea_text_vacio
OK     test_lexer_tokens_basicos
OK     test_lexer_variables
OK     test_lista_no_es_script
OK     test_matriz_16_en_la_etapa_apropiada
OK     test_matriz_16_negativos
OK     test_matriz_16_positivos
OK     test_modos_de_palabra
OK     test_nombre_calculado_fuera_de_perf

### Matriz adicional que debe convertir en pruebas

| Entrada | Resultado esperado / nivel |
|---|---|
| `puts ""` y `puts {}` | Dos palabras; argumento vacío, modos diferentes |
| `set x hola_${nombre}_fin` | Tres palabras; la última tiene literal, variable y literal |
| `puts [set x 1; set x]` | Sustitución con dos comandos |
| `puts [list [set x] y]` | Sustituciones anidadas |
| `puts {a {b} c}` | Literal `a {b} c`, sin comandos internos |
| `puts {a\{b\}c}` | Literal conserva las barras y no altera balanceo |
| `puts a#b` / `puts #dato` | `#` dentro del argumento, sin comentario |
| `# comentario` | Script vacío |
| `puts "a; b"` | `;` dentro del literal, sin separar comandos |
| `puts $a(3)` | Referencia al arreglo `a`, índice literal `3` |
| `puts $a($i)` | Fuera del perfil léxico; error localizado |
| `puts {sin cierre` | Error de llave faltante, señalar apertura |
| `puts "sin cierre` | Error de comillas faltantes |
| `puts [set x 1` | Error de corchete faltante |
| `puts {a}b` / `puts "a"b` | Error por texto después de una palabra agrupada |
| `puts ]` | Error: cierre fuera de contexto |
| `set x 1 2` | AST estructural válido; error de aridad en validación |
| `if {1} {puts si} else` | AST estructural válido; falta cuerpo en validación |
| `expr {2 + }` | Script válido; error al analizar su expresión |
| `expr {2 3}` | Error de expresión por residuos |
| `expr {0 && [error inesperado]}` | AST válido; futuro evaluador no ejecuta `error` |
| `proc f {x x} {return $x}` | Error de parámetros duplicados en validación |
| `puts {if mal formado}` | Válido; ese argumento es texto, no un condicional |

Para errores de entrada, las fachadas públicas deben lanzar `ParseError` (incluidos errores del lexer, o una subclase). No capture cualquier `Exception` en las pruebas negativas: ocultaría errores de programación o actividades aún sin resolver.

## 17. Diagnósticos y estrategia de implementación

Cada diagnóstico debe contener nombre de fuente, línea, columna, elemento esperado y elemento encontrado. El cursor suministrado conserva offsets; **actividad:** calcular línea y columna a partir de la fuente normalizada, y mostrar el fragmento afectado. En un cierre faltante conviene incluir también la posición de apertura.

Orden de trabajo sugerido:

1. Lexer de texto, espacios, separadores y comentarios; comprobar el esqueleto con tokens reales.
2. `BRACED`, variables y palabras con múltiples partes.
3. Comillas y corchetes anidados; probar cambios de contexto y restauración del modo.
4. `ExprLexer(sly.Lexer)` y `ExprParser(sly.Parser)`: tokens, producciones LALR(1), acciones de AST y errores.
5. Validador de formas de comandos y reanálisis de cuerpos y condiciones.
6. Lector de listas, parámetros de procedimientos y casos de integración.
7. Posiciones completas, mensajes y pruebas negativas.

Deténgase en el primer error para esta entrega. La recuperación por sincronización es opcional: no descarte tokens hasta el primer `;` sin considerar comillas y anidamiento.

### Preguntas que deben acompañar la entrega

1. ¿Por qué `set x {$y}` no debe generar una referencia a `y`?
2. ¿Qué evita que `hola_$nombre` se convierta en dos argumentos?
3. ¿Qué parte del sistema distingue `set n 12` de `expr {12}`?
4. ¿Por qué la gramática de SLY admite recursividad izquierda para la suma, mientras la potencia asocia a la derecha?
5. ¿Cómo conserva el análisis de `0 && [error fallo]` la posibilidad de cortocircuito?
6. ¿Qué hace el lexer con `]` dentro de llaves y con `;` dentro de comillas?
7. ¿Qué errores puede detectar el parser genérico y cuáles requieren conocer el comando?
8. ¿Cómo cambian los contextos de `break` y `return` al entrar a un nuevo procedimiento?

## 18. Entregables y criterios de evaluación

Entregar el cuaderno completado, los módulos equivalentes (`model.py`, `script_lexer.py`, `script_parser.py`, `expr_lexer.py`, `expr_parser.py`, `command_analyzer.py`) y pruebas reproducibles. Los nombres de código se escriben en inglés y las explicaciones en español.

| Criterio | Peso |
|---|---:|
| Gramática aplicada y explicación de decisiones predictivas | 15 % |
| Léxico contextual, balanceo y límites de palabras | 20 % |
| Parser estructural y construcción correcta del AST | 25 % |
| Lexer y parser con SLY: AST, precedencia y asociatividad | 20 % |
| Validación de comandos y análisis de cuerpos estáticos | 10 % |
| Diagnósticos y pruebas positivas/negativas | 10 % |

**Condición de terminación:** no quedan `NotImplementedError` en las rutas obligatorias, se puede analizar cada programa de la sección 15 sin ejecutarlo, se obtienen AST verificables y se rechazan los casos inválidos en la etapa apropiada. El intérprete y el generador de código serán trabajos posteriores.

La entrega debe demostrar que `ExprLexer` deriva de `sly.Lexer` y `ExprParser` de `sly.Parser`, que la gramática completa genera tablas sin conflictos no justificados y que sus acciones producen AST. Debe comprobarse que tanto `expr` como las condiciones pasan por la misma fachada SLY. Los scripts se analizan con el lexer manual y descenso recursivo.


## 19. Referencias y decisiones de diseño

Fuentes primarias consultadas para delimitar el lenguaje:

- [Reglas de sintaxis de Tcl](https://www.tcl-lang.org/man/tcl8.7/TclCmd/Tcl.html): separación entre comandos, palabras, agrupación y sustituciones.
- [Picol de antirez](https://github.com/antirez/picol): referencia de intérprete pequeño y didáctico. Existen versiones y derivados con capacidades distintas; no se asume que todos tengan el mismo repertorio.
- [Jim Tcl](https://jim.tcl-lang.org/home/doc/www/www/index.html) y [su documentación](https://jim.tcl-lang.org/home/doc/trunk/jim_tcl.txt): referencia de una implementación con un alcance considerablemente mayor.

La gramática y los contratos de este cuaderno son una **propuesta docente propia**, no una transcripción del manual de Tcl ni una promesa de compatibilidad completa. Los límites de índices, escapes, cuerpos estáticos, `expr`, listas y `catch` son decisiones de este perfil. El tamaño intermedio se busca por el alcance de la biblioteca y la arquitectura; no por una cifra de líneas de código.

Este cuaderno retoma el alcance y la precedencia descritos en la especificación MiniTcl 1.0 del curso y documenta explícitamente sus ampliaciones. Las fuentes externas sirven para contextualizar; el contrato obligatorio para el estudiante es el definido aquí.

- [Documentación oficial de SLY](https://sly.readthedocs.io/en/latest/sly.html): clases `Lexer`/`Parser`, producciones, posiciones y generación LALR(1).


## 20. EBNF para diagramas railroad: dos entradas y un puente

El archivo `MiniTcl_Railroad.ebnf` usa notación de estilo W3C para el generador de Bottlecaps. `manual_script_input` inicia la gramática del lexer manual y el parser descendente recursivo. `sly_expression_input` inicia la gramática LALR(1) de SLY; su fin de entrada es implícito.

Los diagramas de `builtin_forms` son contratos de validación, no alternativas nuevas del parser estructural. `expr_argument`, `test` y `body` aparecen como `BRACED`: el análisis de comandos entrega el contenido de los dos primeros a SLY y el de `body` al parser manual. Es una delegación entre analizadores, no la inserción de la gramática de expresiones dentro del lexer de scripts.

`CMD_SUBST` representa un token SLY cuyo valor es un AST `Script`; el patrón de caracteres `[...]` se muestra en `substitution`, dentro de la gramática manual. Las reglas auxiliares `variable_syntax`, `number_syntax` y `boolean_syntax` explican la forma del texto; no son producciones adicionales de SLY ni sustituyen sus expresiones regulares. Los nombres de tokens entre comillas son etiquetas para el diagrama.

Las producciones de expresiones conservan recursividad izquierda intencionadamente: son para SLY. Las de scripts conservan repeticiones para facilitar el descenso recursivo.

Copie el siguiente bloque en https://www.bottlecaps.de/rr/ui:

```ebnf
manual_script_input ::= program
program ::= script 'EOF'

script ::= ('WS' | 'SEP')* (command (command_gap command)*)? ('WS' | 'SEP')*
command_gap ::= 'WS'* 'SEP' ('WS' | 'SEP')*
command ::= word ('WS' word)*
word ::= 'BRACED' | quoted_word | bare_word
quoted_word ::= 'QOPEN' part* 'QCLOSE'
bare_word ::= part+
part ::= 'TEXT' | 'VAR' | substitution
substitution ::= '[' script ']'

sly_expression_input ::= expression
expression ::= logical_or
logical_or ::= logical_or '||' logical_and | logical_and
logical_and ::= logical_and '&&' equality | equality
equality ::= equality ('==' | '!=' | 'eq' | 'ne') relational | relational
relational ::= relational ('<' | '<=' | '>' | '>=') additive | additive
additive ::= additive ('+' | '-') multiplicative | multiplicative
multiplicative ::= multiplicative ('*' | '/' | '%') unary | unary
unary ::= ('+' | '-' | '!') unary | power
power ::= primary '**' unary | primary
primary ::= 'NUMBER' | 'BOOL' | 'VAR' | 'STRING' | '(' expression ')' | 'CMD_SUBST'

builtin_forms ::= set_cmd | unset_cmd | incr_cmd | append_cmd | puts_cmd
                | expr_cmd | if_cmd | while_cmd | for_cmd | foreach_cmd
                | proc_cmd | return_cmd | break_cmd | continue_cmd
                | global_cmd | catch_cmd | error_cmd
                | list_cmd | llength_cmd | lindex_cmd | lappend_cmd | lset_cmd
                | dict_cmd | string_cmd | split_cmd | join_cmd

set_cmd ::= 'set' 'WS' name ('WS' word)?
unset_cmd ::= 'unset' 'WS' name
incr_cmd ::= 'incr' 'WS' name ('WS' word)?
append_cmd ::= 'append' 'WS' name ('WS' word)+
puts_cmd ::= 'puts' 'WS' ('-nonewline' 'WS')? word
expr_cmd ::= 'expr' 'WS' expr_argument
if_cmd ::= 'if' 'WS' test ('WS' 'then')? 'WS' body
           ('WS' 'elseif' 'WS' test ('WS' 'then')? 'WS' body)*
           ('WS' 'else' 'WS' body)?
while_cmd ::= 'while' 'WS' test 'WS' body
for_cmd ::= 'for' 'WS' body 'WS' test 'WS' body 'WS' body
foreach_cmd ::= 'foreach' 'WS' identifier 'WS' word 'WS' body
proc_cmd ::= 'proc' 'WS' identifier 'WS' parameters 'WS' body
return_cmd ::= 'return' ('WS' word)?
break_cmd ::= 'break'
continue_cmd ::= 'continue'
global_cmd ::= 'global' ('WS' identifier)+
catch_cmd ::= 'catch' 'WS' body ('WS' identifier)?
error_cmd ::= 'error' 'WS' word
call_cmd ::= word ('WS' word)*

body ::= 'BRACED'
expr_argument ::= 'BRACED'
test ::= expr_argument
parameters ::= 'BRACED'
parameter_content ::= 'LIST_WS'* (identifier ('LIST_WS' identifier)*)? 'LIST_WS'*
name ::= identifier ('(' index ')')?

list_cmd ::= 'list' ('WS' word)*
llength_cmd ::= 'llength' 'WS' word
lindex_cmd ::= 'lindex' 'WS' word 'WS' word
lappend_cmd ::= 'lappend' 'WS' name ('WS' word)*
lset_cmd ::= 'lset' 'WS' name 'WS' word 'WS' word
dict_cmd ::= dict_create_cmd | dict_get_cmd | dict_exists_cmd | dict_set_cmd
dict_create_cmd ::= 'dict' 'WS' 'create' ('WS' word 'WS' word)*
dict_get_cmd ::= 'dict' 'WS' 'get' 'WS' word 'WS' word
dict_exists_cmd ::= 'dict' 'WS' 'exists' 'WS' word 'WS' word
dict_set_cmd ::= 'dict' 'WS' 'set' 'WS' name 'WS' word 'WS' word
string_cmd ::= string_length_cmd | string_case_cmd | string_compare_cmd
string_length_cmd ::= 'string' 'WS' 'length' 'WS' word
string_case_cmd ::= 'string' 'WS' ('tolower' | 'toupper') 'WS' word
string_compare_cmd ::= 'string' 'WS' 'compare' 'WS' word 'WS' word
split_cmd ::= 'split' 'WS' word ('WS' word)?
join_cmd ::= 'join' 'WS' word ('WS' word)?

list_value ::= 'LIST_WS'* (element ('LIST_WS' element)*)? 'LIST_WS'*
element ::= 'LIST_BRACED' | 'LIST_QUOTED' | 'LIST_BARE'

variable_syntax ::= '$' identifier ('(' index ')')? | '${' identifier '}'
identifier ::= letter_or_underscore (letter_or_underscore | digit)*
letter_or_underscore ::= [A-Z] | [a-z] | '_'
digit ::= [0-9]
index ::= index_char+
index_char ::= letter_or_underscore | digit | '.' | '-' | ':'
number_syntax ::= (digit+ ('.' digit*)? | '.' digit+) exponent?
exponent ::= ('e' | 'E') ('+' | '-')? digit+
boolean_syntax ::= 'true' | 'false' | 'yes' | 'no' | 'on' | 'off'
```

## 21. Entrega: módulos, pruebas y respuestas

El cuaderno es la fuente de verdad de la entrega.  Cada celda de código lleva la marca `# === MODULE: <fichero> | PARTE n/m ===`; `tools/export_modules.py` reconstruye los módulos a partir del cuaderno, de modo que **cuaderno y módulos no pueden divergir**.

| Módulo | Contenido |
|---|---|
| `model.py` | AST (`Span`, `Node`, `Word`, `Script`, `Expr`…), `Token`, `ParseError` con línea y columna, `cover()`, `expr_shape()` y el `TokenCursor`. |
| `script_lexer.py` | Lexer manual contextual: modos comando/llaves/comillas, escapes, variables, balanceo, comentarios y `scan_substitution()`. |
| `script_parser.py` | `ScriptParser` descendente recursivo, `parse_minitcl()` y el callback `read_substitution()`. |
| `list_reader.py` | Lector de listas (`LIST_WS`, `LIST_BRACED`, `LIST_QUOTED`, `LIST_BARE`) y `format_list()`. |
| `expr_lexer.py` | `ExprLexer(sly.Lexer)` con los tokens de la sección 5 y la regla `CMD_SUBST` que delega en el lexer manual. |
| `expr_parser.py` | `ExprParser(sly.Parser)` con la gramática por niveles y `parse_minitcl_expr()`. |
| `command_analyzer.py` | Validación de las formas de comando y reanálisis estático de cuerpos y condiciones. |
| `tests/test_minitcl.py` | Pruebas reproducibles (no requieren `pytest`). |
| `demo.py` | Demostración del pipeline en la terminal: `python3 demo.py [fichero.tcl]`. |
| `ejemplos/` | Los cuatro programas de aceptación de la sección 15 y uno con un error. |
| `Makefile` | El ciclo de trabajo: `make test`, `make demo`, `make build`, `make run`, `make check`. |

Ciclo de trabajo (`make help` lista todos los objetivos):

```bash
make test                                  # las 35 pruebas
make demo                                  # la demostración en la terminal
make demo ARCHIVO=ejemplos/a_factorial.tcl   # con un fichero MiniTcl
make build                                 # regenera el cuaderno y los módulos
make run                                   # ejecuta el cuaderno y guarda las impresiones
make check                                 # build + test + run (el ciclo completo)
```

`make build` va siempre **antes** que `make run`: regenerar el cuaderno borra las impresiones que había guardado.


### 21.1  Actividad de la sección 11: por qué el modo del lexer es parte del contrato

El significado de un carácter **depende del contexto**, así que el modo no es un detalle de implementación del lexer sino parte del contrato léxico:

| Contexto | `{` | `}` | `"` | `$` | `[` | `]` | `#` | `;` |
|---|---|---|---|---|---|---|---|---|
| palabra desnuda | error | error | error | abre variable | abre sustitución | termina la palabra | texto | separador |
| entre llaves | texto | texto | texto | texto | texto | texto | texto | texto |
| entre comillas | texto | texto | cierra | abre variable | abre sustitución | texto | texto | texto |
| inicio de palabra | abre bloque | — | abre palabra | abre variable | abre sustitución | error fuera de contexto | comentario si empieza un comando | separador |

Una expresión regular aislada (o una lista plana de tokens) **no** puede expresar esa tabla: el valor de `#` depende de si la posición permite empezar un comando, y el de `}` depende del balanceo de llaves.  Por eso el lexer mantiene una pila de marcos (`command` / `quote`) y por eso el parser descendente recibe `BRACED`, `QOPEN` y `QCLOSE` como tokens ya decididos: nunca vuelve a examinar su contenido.

No se afirma que toda la sintaxis de Tcl sea una gramática LL(1) sobre caracteres: el análisis léxico es contextual (con pila de modos) y sólo después de tokenizar, la gramática estructural es LL(1) sobre esos tokens.


### 21.2  Actividad de la sección 12: recursividad izquierda y AST

**Asociación a la izquierda.**  `additive : additive MINUS multiplicative` es recursiva por la izquierda: el parser resuelve `10 - 3` y, antes de reducir `additive`, ve que el siguiente token es `-`; entonces desplaza el `additive` ya leído a la izquierda y continúa.  La reducción sólo ocurre cuando la mirada no es `+` ni `-`, de modo que `10 - 3 - 2` se agrupa como `(10 - 3) - 2`.

**Asociación a la derecha.**  `power : primary POWER unary` es recursiva por la derecha: el operando derecho se analiza como un `unary` completo, que puede ser a su vez otro `power`; así `2 ** 3 ** 2` se agrupa como `2 ** (3 ** 2)`.

**Precedencia entre unario y potencia.**  `unary : MINUS unary | power` y `power : primary POWER unary | primary`.  El token `**` sólo puede seguir a un `primary`, de modo que en `-2 ** 2` el `**` se une al `primary` antes de que el `unary` se reduzca: `-(2 ** 2)`.  En `2 ** -3` el `**` ya está consumido y el operando derecho es `unary -> MINUS unary`: `2 ** (-3)`.

Los AST que se pedían dibujar antes de evaluarlos:

```
8 / 2 * 3         ->  Binary('*', Binary('/', 8, 2), 3)
2 ** 3 ** 2       ->  Binary('**', 2, Binary('**', 3, 2))
-(4 + 1)          ->  Unary('-', Binary('+', 4, 1))
$a(3) + [doble 2] ->  Binary('+', VariableExpr(Variable('a', '3')),
                              SubstitutionExpr(Script(...)))
```

Los paréntesis de agrupación no crean nodo propio: su contenido se devuelve tal cual.


### 21.3  Actividad de la sección 13: `analyze_command`

La segunda pasada vive en `command_analyzer.py` y **no sustituye** al AST estructural: lo anota.

```python
@dataclass(frozen=True)
class ArgumentAnalysis:
    word_index: int          # índice de la palabra dentro del comando
    kind: str                # 'expr' | 'script' | 'list' | 'name' |
                              # 'identifier' | 'word'
    span: Span
    value: object = None     # Expr, Script, tuple[str] o Word

@dataclass(frozen=True)
class CommandAnalysis:
    command: Command         # el comando estructural, intacto
    form: str                # 'set_cmd', 'if_cmd', 'call_cmd', ...
    arguments: tuple[ArgumentAnalysis, ...]
    substitutions: tuple[ScriptAnalysis, ...]   # los [ ... ] del comando
    bodies: tuple[ScriptAnalysis, ...]          # los cuerpos reanalizados
```

`analyze_command(command, source, source_name)` devuelve un `CommandAnalysis`; `analyze_script(script, ...)` devuelve un `ScriptAnalysis` con el análisis de cada comando, y `analyze_minitcl(source)` es la fachada que analiza y valida.

Reglas que aplica:

* La **forma** se comprueba contra la tabla de la sección 6 (aridad y estructura).  `call_cmd` es la forma general y no se prueba después de fallar con un incorporado.
* El **nombre** del comando debe ser literal; un nombre calculado (`$accion hola`) es estructuralmente válido pero el validador lo rechaza como fuera del perfil.
* Las **condiciones** (`expr`, `if`, `elseif`, `while`, `for`) se entregan a `parse_minitcl_expr`; los **cuerpos** de `if`, `elseif`, `while`, `for`, `foreach`, `proc` y `catch` se entregan a `parse_minitcl`.  Los dos caminos comparten la misma fachada de expresiones: no existe un segundo parser de condiciones.
* Al reanalizar un `BRACED`, la posición interior es `word.span.start + 1`; ese desplazamiento se propaga a los spans y al nombre de la fuente, de modo que las posiciones no se calculan sobre texto interpolado.
* Los bloques de `set`, `puts`, `list` o los datos de `dict` **no** se interpretan como scripts.
* Los corchetes ya producen un `Script` durante el análisis estructural: el validador los visita también.
* Un error de sintaxis dentro de un cuerpo estático se reporta aunque la rama no se tome: el cuerpo se analiza y se valida siempre.


### 21.4  Actividad de la sección 15: invocaciones recursivas de análisis

Se analiza el programa A:

```tcl
proc factorial {n} {
    if {$n <= 1} {return 1}
    return [expr {$n * [factorial [expr {$n - 1}]]}]
}
puts [factorial 6]
```

Invocaciones de análisis, en orden:

1. `parse_minitcl(programa)` — el script completo.
2. El comando `proc`: su **cuerpo** `BRACED` se reanaliza con `parse_minitcl` (base `+1`), y dentro de ese cuerpo:
3. El comando `if`: su **condición** `{$n <= 1}` se entrega a `parse_minitcl_expr` y su **cuerpo** `{return 1}` a `parse_minitcl`.
4. El comando `return`: su argumento `[expr {...}]` ya es un `Script` construido por el parser manual; el validador lo visita y dentro aparece el comando `expr`, cuyo argumento `{$n * [factorial [expr {$n - 1}]]}` se entrega a `parse_minitcl_expr`.
5. Esa expresión contiene otra sustitución: el lexer de SLY llama a `read_substitution`, que vuelve al **parser manual** y produce el `Script` del comando `factorial`; su argumento `[expr {$n - 1}]` es otra sustitución, y su expresión `{$n - 1}` se entrega de nuevo a `parse_minitcl_expr`.
6. El comando `puts`: su argumento `[factorial 6]` es un `Script` estructural que también se visita.

Las palabras cuyo contenido debe analizarse de nuevo son exactamente las **palabras `BRACED`** con un papel definido por la forma del comando: el cuerpo de `proc`, la condición y el cuerpo de `if`, y el argumento de cada `expr` (incluidos los que aparecen dentro de una sustitución).  Las palabras `puts`, `proc`, `factorial`, `return` e `if` son nombres literales de comando y no se reanalizan; `n` es una variable de la expresión y se resuelve en tiempo de ejecución, no de análisis.


### 21.5  Actividad de la sección 17: línea y columna

Los offsets del `Span` se refieren a la fuente **normalizada** (CRLF y CR convertidos a LF antes de lexear).  Para un offset `o`:

```python
line = source.count("
", 0, o) + 1
line_start = source.rfind("
", 0, o) + 1
column = o - line_start + 1
```

Eso es exactamente lo que hace `model.position_of`.  `model.format_diagnostic` compone el mensaje con fuente, línea y columna, y añade el fragmento afectado con un cursor (`^`) debajo.  `ParseError` guarda `line`, `column` y `span`, y el `TokenCursor` acepta la fuente y el desplazamiento base para que las posiciones sigan siendo correctas al reanálisis de un bloque interior (`base_offset = word.span.start + 1`).

Para un cierre faltante, el error del lexer se produce en la posición de **apertura** (`{`, `"` o `[`), que es donde el lector puede localizar el problema.


### 21.6  Las ocho preguntas del cuaderno

**1. ¿Por qué `set x {$y}` no debe generar una referencia a `y`?**

Porque una palabra entre llaves es literal: el lexer cuenta las llaves anidadas y devuelve su contenido interior como un único `Literal`, sin interpretar `$`, `[` ni escapes.  Sólo las palabras desnudas y entre comillas contienen sustituciones.  El validador tampoco reanaliza el valor de `set`, porque la forma `set_cmd` no lo exige.

**2. ¿Qué evita que `hola_$nombre` se convierta en dos argumentos?**

El lexer emite el fragmento `TEXT` `hola_` y el token `VAR` `$nombre` **sin** ningún `WS` ni `SEP` entre ellos, y `parse_bare_word` acumula partes contiguas (`TEXT | VAR | LBRACKET`) en una sola `Word`.  Los límites de palabra sólo los crean los espacios en blanco y los separadores.

**3. ¿Qué parte del sistema distingue `set n 12` de `expr {12}`?**

La validación de comandos (segunda pasada).  Los dos son, para el parser estructural, un comando con dos palabras.  `set_cmd` exige un nombre literal y un valor que **no** se reanaliza; `expr_cmd` exige exactamente una palabra entre llaves, y su contenido se entrega a `parse_minitcl_expr`, que lo analiza con SLY.

**4. ¿Por qué la gramática de SLY admite recursividad izquierda para la suma, mientras la potencia asocia a la derecha?**

Porque está escrito así: `additive : additive PLUS multiplicative` deja el operando izquierdo ya construido y permite continuar con otro `+`/`-` antes de reducir (asociación izquierda), mientras que `power : primary POWER unary` construye primero el operando derecho como un `unary` completo, que puede ser otro `power` (asociación derecha).  El autómata LALR(1) que genera SLY admite ambas formas y los niveles de la gramática dan la precedencia sin tabla `precedence` adicional.

**5. ¿Cómo conserva el análisis de `0 && [error fallo]` la posibilidad de cortocircuito?**

Porque el análisis nunca evalúa.  La sustitución sólo se **analiza**: el lexer manual construye un `Script` y el parser de SLY lo envuelve en un `SubstitutionExpr`.  El AST queda como `Binary('&&', 0, SubstitutionExpr(...))` y la decisión de no evaluar el operando derecho cuando el izquierdo es falso queda en manos del evaluador futuro.  Ningún paso consulta el valor de `x` ni ejecuta el cuerpo de la sustitución.

**6. ¿Qué hace el lexer con `]` dentro de llaves y con `;` dentro de comillas?**

Dentro de las llaves, `]` es texto: el balanceo se cuenta con llaves, no con corchetes.  Dentro de las comillas, `;` forma parte del `TEXT` y no separa comandos; sólo `"` cierra la palabra entre comillas.  En cambio, un `]` en el nivel exterior es un error de cierre fuera de contexto.

**7. ¿Qué errores puede detectar el parser genérico y cuáles requieren conocer el comando?**

El parser genérico detecta los errores léxicos (llaves, comillas o corchetes sin cerrar, escapes desconocidos, `$` sin nombre, `]` fuera de contexto, texto tras una palabra agrupada) y los estructurales (falta un separador entre comandos, token inesperado al empezar una palabra, `]` que no cierra).  No puede detectar los errores de forma: `set x 1 2`, `if {1} {puts si} else` o `proc f {x x} {...}` son AST estructurales válidos y los rechaza la segunda pasada, que sí conoce el comando.

**8. ¿Cómo cambian los contextos de `break` y `return` al entrar a un nuevo procedimiento?**

`return` sólo es válido dentro de un procedimiento: al invocar uno se crea un ámbito local nuevo, y el contexto "estoy en un procedimiento" debe establecerse para el cuerpo.  `break` y `continue` requieren un ciclo **en el procedimiento actual** y no pueden escapar al ciclo del llamador, así que al entrar en un procedimiento el contexto de ciclos se reinicia (se guarda y se restaura).  En esta entrega sólo se analiza la forma sintáctica; el análisis estático de un cuerpo crea precisamente ese contexto nuevo, y por eso un `return` fuera de un procedimiento, o un `break` sin ciclo en el procedimiento actual, se detectan en la segunda pasada.
